# P5 — Opioid Dispensing Outcome Construction & Validation

**Project:** Predicting Opioid Dispensing in Low Back Pain Emergency Visits: Explainable Machine Learning and Clinical Phenotype Sensitivity

**Course:** AI in Healthcare — High-Risk Project  
**Dataset:** MIMIC-IV-ED v2.2  
**Unit of Analysis:** Emergency Department Encounter (`stay_id`)  
**Medication Classification Reference:** Cumulus Opioid Value Set (Wasz et al., 2026, *JAMIA Open*)  
**Reference File:** `codebooks/references/opioid_valueset.tsv`  
**Reference Source:** https://github.com/smart-on-fhir/cumulus-library-opioid-valueset-supplement  
**Reference Version:** GitHub commit `f2566d167c74cb9ed3dc21a41581f304ae41af3e`  
**Reference SHA-256:** `33eae9e60fa6584ed77e74bdf9a438dc8202eb161318af446d93d57cf55cf327`  
**Retrieval Date:** November 23, 2026
**Outcome:** Documented dispensing of a Cumulus-defined opioid-related medication (`opioid_dispensed`)  
**Input:** Frozen P4 Encounter-Level Dataset  
**Notebook:** `05_opioid_dispensing_outcome.ipynb`

---
## Executive Summary

This notebook constructs and validates an encounter-level binary opioid dispensing outcome for adult emergency department (ED) visits belonging to the previously established ICD-defined low back pain (LBP) study population.

The analytical population was finalized in P4 and consists of **14,293 ED encounters from 11,404 unique patients**, including 9,571 encounters in the strict LBP cohort (C1) and 4,722 additional encounters identified by the expanded phenotype (C2-only).

Opioid-related medications will be identified using the published **Cumulus Opioid Value Set** (Wasz et al., 2026), an externally developed, RxNorm-based medication terminology reference. A frozen snapshot of the value set will be used to support reproducible classification.

Pyxis medication names and available GSN identifiers will be matched through a focused, auditable terminology-mapping workflow. Ambiguous or unresolved mappings will receive targeted review without developing independent clinical opioid inclusion rules.

The primary outcome, `opioid_dispensed`, will indicate whether at least one qualifying Cumulus-defined medication dispensing record is documented within the finalized ED encounter time window. This outcome does not establish confirmed administration, opioid consumption, treatment indication, or clinical appropriateness.

The workflow includes medication mapping, temporal attribution, encounter-level aggregation, outcome validation, and cohort-specific prevalence analysis. The final dataset will preserve all 14,293 P4 encounters and append one binary outcome variable for predictive modeling in P6.

**Research Boundary:** P5 constructs and validates the outcome only. Cohort membership, ICD phenotypes, and triage-time candidate predictors remain unchanged.


## Table of Contents

- [5.1 Study Objective and P4 Input Validation](#51-study-objective-and-p4-input-validation)
- [5.2 Pyxis Source Review and Medication Inventory](#52-pyxis-source-review-and-medication-inventory)
- [5.3 Standardized Opioid Medication Mapping](#53-standardized-opioid-medication-mapping)
- [5.4 Temporal Attribution and Outcome Construction](#54-temporal-attribution-and-outcome-construction)
- [5.5 Outcome Validation](#55-outcome-validation)
- [5.6 Final Summary and P6 Handoff](#56-final-summary-and-p6-handoff)


## 5.1 Study Objective and P4 Input Validation

### Objective

Establish the frozen P4 encounter-level dataset as the analytical foundation for constructing the opioid dispensing outcome.

The P4 dataset contains 14,293 eligible adult ED encounters from 11,404 unique patients, including two nested ICD-defined low back pain (LBP) phenotypes:

- **C1 — Strict LBP:** 9,571 encounters.
- **C2-only — Expanded-only LBP:** 4,722 encounters.
- **C2 — Expanded LBP:** 14,293 encounters, including all C1 encounters.

P5 will preserve the existing analytical population and append a binary outcome variable, `opioid_dispensed`, derived from MIMIC-IV-ED Pyxis records.

### Method

1. Load the finalized P4 encounter-level dataset.
2. Verify the expected dataset dimensions and identifier integrity.
3. Confirm adult eligibility and nested cohort membership.
4. Verify that no opioid dispensing outcome has already been constructed.
5. Establish the input data contract for subsequent outcome integration.

The P4 cohort definitions, eligibility criteria, and existing predictors will remain unchanged.

### Expected Output

A validated P4 input dataset satisfying the following conditions:

| Validation Metric | Expected Value |
|---|---:|
| ED encounters | 14,293 |
| Unique patients | 11,404 |
| Columns | 23 |
| C1 encounters | 9,571 |
| C2-only encounters | 4,722 |
| C2 encounters | 14,293 |
| Duplicate `stay_id` | 0 |
| Missing identifiers | 0 |
| Adult eligibility | All estimated ages ≥18 |
| Existing opioid outcome | Absent |

Successful validation confirms that the frozen P4 dataset is ready for opioid outcome construction without redefining or excluding encounters.

In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------
# Global project setup
# --------------------------------------------------

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# External medication terminology reference
CUMULUS_PATH = (
    PROJECT_ROOT
    / "codebooks"
    / "references"
    / "opioid_valueset.tsv"
)

P4_PATH = (
    PROJECT_ROOT
    / "results"
    / "04_cohort_integration"
    / "adult_lbp_encounter_features.csv"
)

assert P4_PATH.exists(), f"File not found: {P4_PATH}"



# Load frozen P4 dataset
df_p4 = pd.read_csv(
    P4_PATH,
    dtype={"stay_id": "string", "subject_id": "string"}
)

# --------------------------------------------------
# P4 input contract
# --------------------------------------------------

EXPECTED_ROWS = 14_293
EXPECTED_PATIENTS = 11_404
EXPECTED_COLUMNS = 23
EXPECTED_C1 = 9_571
EXPECTED_C2_ONLY = 4_722

required_columns = {
    "stay_id",
    "subject_id",
    "estimated_age",
    "c1_flag",
    "c2_flag",
    "c2_only_flag",
}

assert required_columns.issubset(df_p4.columns)

assert df_p4.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS)
assert df_p4["stay_id"].is_unique
assert df_p4[["stay_id", "subject_id"]].notna().all().all()
assert (df_p4["estimated_age"] >= 18).all()
assert df_p4["subject_id"].nunique() == EXPECTED_PATIENTS

# Validate cohort flags
flags = df_p4[["c1_flag", "c2_flag", "c2_only_flag"]]

assert flags.notna().all().all()
assert flags.isin([0, 1]).all().all()

assert (df_p4["c2_flag"] == 1).all()
assert (
    df_p4["c1_flag"] + df_p4["c2_only_flag"] == 1
).all()

assert df_p4["c1_flag"].sum() == EXPECTED_C1
assert df_p4["c2_only_flag"].sum() == EXPECTED_C2_ONLY

# Outcome must not exist in P4
assert "opioid_dispensed" not in df_p4.columns

# --------------------------------------------------
# Validation summary
# --------------------------------------------------

summary = pd.DataFrame({
    "Metric": [
        "Total encounters",
        "Unique patients",
        "Total columns",
        "C1 encounters",
        "C2-only encounters",
        "C2 encounters",
        "Missing stay_id",
        "Duplicate stay_id",
    ],
    "Observed": [
        len(df_p4),
        df_p4["subject_id"].nunique(),
        df_p4.shape[1],
        int(df_p4["c1_flag"].sum()),
        int(df_p4["c2_only_flag"].sum()),
        int(df_p4["c2_flag"].sum()),
        int(df_p4["stay_id"].isna().sum()),
        int(df_p4["stay_id"].duplicated().sum()),
    ],
})

display(summary)

print("P4 input validation: PASSED")


,Metric,Observed
0,Total encounters,14293
1,Unique patients,11404
2,Total columns,23
3,C1 encounters,9571
4,C2-only encounters,4722
5,C2 encounters,14293
6,Missing stay_id,0
7,Duplicate stay_id,0


P4 input validation: PASSED


### Actual Results

The frozen P4 encounter-level dataset was successfully loaded and passed all input validation checks.

| Validation Metric | Observed | Status |
|---|---:|---|
| Total encounters | 14,293 | Pass |
| Unique patients | 11,404 | Pass |
| Total columns | 23 | Pass |
| C1 encounters | 9,571 | Pass |
| C2-only encounters | 4,722 | Pass |
| C2 encounters | 14,293 | Pass |
| Missing `stay_id` | 0 | Pass |
| Duplicate `stay_id` | 0 | Pass |

Additional assertions confirmed valid adult eligibility, complete patient identifiers, binary cohort flags, correct C1/C2 nesting, and the absence of a pre-existing `opioid_dispensed` field.

**Validation Status: PASSED**

### Findings

The P4 dataset satisfies the prespecified input contract and provides a stable analytical population for P5 outcome construction.

All 14,293 encounters are retained, with exactly one record per `stay_id`. The nested cohort structure is consistent with the finalized ICD phenotype definitions.

No additional cohort filtering or reconstruction is required.

The subsequent Pyxis integration will use this validated dataset as the fixed analytical base. Encounters without qualifying opioid dispensing records will remain in the population.

### Conclusion

The P4 analytical population is validated and ready for opioid dispensing outcome construction. No changes to cohort membership, existing features, or eligibility criteria are necessary.

## 5.2 Pyxis Source Review and Medication Inventory

### 5.2.1 Review of Existing Pyxis Evidence

**Objective**

Review previously validated MIMIC-IV-ED Pyxis source-data findings to establish the requirements for opioid medication identification and encounter-level outcome construction.

This review reuses the database-wide exploratory analysis completed in P1 rather than repeating source-table EDA.

**Method**

Review the existing P1 evidence covering:

1. Pyxis record granularity and encounter linkage.
2. Referential integrity of medication records.
3. ED encounter timestamp validity.
4. Pyxis event timing relative to ED arrival and departure.
5. Data limitations relevant to medication classification and outcome attribution.

**Source Notebook:** `notebooks/01_eda_mimiciv_ed.ipynb`

**Reused SQL Scripts:**

- `sql/01_eda/eda_01_schema.sql`
- `sql/01_eda/eda_02_table_granularity.sql`
- `sql/01_eda/eda_04_referential_integrity.sql`
- `sql/01_eda/eda_12_ed_stay_duration.sql`
- `sql/01_eda/eda_13_event_timestamp_validation.sql`
- `sql/01_eda/eda_14_post_departure_offsets.sql`

**Expected Output**

A concise summary of established Pyxis source characteristics and the remaining cohort-specific information required for standardized opioid outcome construction.

#### Actual Results

The P1 database-wide analysis established the following:

| Domain | Verified P1 Finding |
|---|---|
| Total Pyxis records | 1,586,053 |
| Encounters with Pyxis records | 303,361 |
| Null or unmatched Pyxis `stay_id` | 0 |
| Pyxis records within ED interval | 90.88% |
| Pyxis records before ED arrival | 0.05% |
| Pyxis records after ED departure | 9.07% |
| Missing Pyxis event timestamps | 0 |
| Post-departure Pyxis records | 143,854 |
| Encounters affected by post-departure Pyxis records | 26,702 |

Among post-departure Pyxis records, 68.16% were charted between 1 and 24 hours after recorded ED departure.

The ED encounter duration analysis also identified six negative-duration and 53 zero-duration encounters across the full ED database.

These results describe the source database and are not specific to the finalized adult LBP cohorts.

#### Findings

**1. Encounter linkage**

Pyxis contains multiple medication-related records per ED encounter. P1 found no null or unmatched `stay_id` values, supporting encounter-level linkage. Medication records must be aggregated before integration with the frozen P4 dataset.

**2. Temporal attribution**

Most Pyxis records fall within the recorded ED interval, but a substantial minority occur after departure. The observed post-departure offsets extend beyond short documentation delays.

Therefore, P5 must explicitly validate temporal attribution among qualifying opioid records before assigning encounter-level outcomes.

**3. Medication classification**

The presence of any Pyxis record does not establish opioid dispensing. Medication identification will follow a frozen version of the Cumulus.

The existing P1 summaries do not provide a complete medication-name inventory for the P4 cohort. A focused, cohort-linked inventory is therefore required before terminology mapping.

**4. Source-data limitations**

Pyxis documentation does not establish confirmed medication administration or patient consumption. Timestamp inconsistencies and medication mapping uncertainty remain potential sources of outcome misclassification.

#### Conclusion

Existing P1 results establish the structural and temporal characteristics needed to plan P5. No additional database-wide Pyxis EDA is required.

The next step is to construct a focused Pyxis medication inventory restricted to the 14,293 frozen P4 encounters, preserving medication names, available identifiers, and record frequencies for subsequent Cumulus mapping.

### 5.2.2 P4-Linked Pyxis Medication Inventory

**Objective**

Construct a focused medication inventory for the frozen P4 adult LBP cohort to support subsequent Cumulus-based opioid identification.

**Method**

1. Use the validated BigQuery table `p4_adult_cohort_lookup` to identify the 14,293 eligible ED encounters.
2. Join the cohort to MIMIC-IV-ED Pyxis using `stay_id`.
3. Aggregate medication records by the original `(name, gsn)` combination.
4. Calculate medication record frequencies and unique encounter counts.
5. Summarize medication timestamp placement relative to the recorded ED interval.

The original medication names and GSN identifiers will be preserved without applying opioid-specific classification rules.

**Expected Output**

A medication-level inventory containing:

- Original Pyxis medication name and GSN.
- Number of Pyxis source records.
- Number of distinct ED encounters.
- Record counts before, during, and after the documented ED interval.
- Missing timestamp counts.

This inventory will provide the source data for focused RxNorm terminology matching and Cumulus value set membership verification in Section 5.3.

#### SQL Execution Record

**Script:** `sql/05_opioid_outcome/02_pyxis_medication_inventory.sql`  
**Output:** `results/05_opioid_outcome/pyxis_medication_inventory.csv`  
**Source:** MIMIC-IV-ED Pyxis, linked to the frozen P4 adult cohort  
**Output Granularity:** Unique `(pyxis_name, pyxis_gsn)` combination

#### Actual Results

The cohort-linked Pyxis inventory contained **54,814 medication records**, represented by 1,034 unique medication name–GSN combinations, 712 unique medication names, and 518 distinct non-null GSN identifiers.

| Data Quality Metric | Observed |
|---|---:|
| Total Pyxis records | 54,814 |
| Unique name–GSN combinations | 1,034 |
| Unique medication names | 712 |
| Distinct non-null GSN | 518 |
| Records missing medication name | 0 |
| Records missing GSN | 334 (0.61%) |
| Records missing charttime | 0 |
| Records within ED interval | 51,029 (93.10%) |
| Records before ED arrival | 18 (0.03%) |
| Records after ED departure | 3,767 (6.87%) |
| Unclassified timing records | 0 |

The timing categories were mutually exclusive and collectively accounted for all 54,814 source records.

#### Findings

**1. Medication inventory and terminology variation**

The inventory contains substantial medication-name variation, including generic names, brand references, combination products, dose-specific descriptions, and abbreviated or truncated local naming conventions.

Some medication names correspond to multiple GSN identifiers, while individual GSN identifiers may appear alongside multiple source names.

The combination of medication name and GSN will therefore be retained for mapping traceability. Medication row counts will not be interpreted as counts of independent dispensing events.

**2. Medication identifier completeness**

GSN identifiers were missing from 334 records (0.61%), corresponding to 28 unique medication names. Medication names and chart timestamps were complete in the extracted inventory.

The relatively limited GSN missingness supports a focused terminology-mapping strategy, although identifier availability does not establish mapping accuracy.

**3. Temporal attribution**

Most Pyxis records (93.10%) were charted within the documented ED interval. However, 3,767 records (6.87%) occurred after recorded ED departure, and 18 (0.03%) occurred before ED arrival.

These observations support using a prespecified ED encounter window and evaluating boundary discrepancies among Cumulus-qualified opioid records before finalizing the binary outcome.

**4. Mapping readiness**

The inventory is sufficiently focused to support medication-name matching and targeted terminology review without requiring comprehensive normalization of every source medication.

The next phase will use a frozen Cumulus *Opioids, All* value set as the classification standard. Local medication names and GSN identifiers will support terminology matching, while ambiguous or potentially missed opioid medications will undergo targeted review.

No medication has yet been classified as a confirmed Cumulus opioid member.

#### Conclusion

The P4-linked Pyxis medication inventory has been successfully constructed and reviewed. Source medication names, identifiers, record frequencies, and timing categories are available for subsequent standardized opioid identification.

The results support proceeding with a focused mapping workflow while preserving explicit checks for terminology uncertainty and temporal misclassification.

### 5.2.3 Source Data Quality and Mapping Readiness

**Objective**

Validate the exported P4-linked Pyxis medication inventory and assess its suitability for the focused Cumulus-based medication mapping workflow in Section 5.3.

This section evaluates source-data completeness, medication identifier consistency, and the distribution of medication records without performing opioid classification.

**Method**

1. Load the medication inventory exported in Section 5.2.2.
2. Verify uniqueness of `(pyxis_name, pyxis_gsn)` combinations and the integrity of aggregated record counts.
3. Assess medication-name and GSN completeness.
4. Examine the concentration of records among frequently observed medication combinations.
5. Identify source terminology characteristics relevant to subsequent mapping and targeted review.

GSN values will be treated as identifiers rather than numerical quantities. Original medication names will be preserved.

**Source Data**

- SQL: `sql/05_opioid_outcome/02_pyxis_medication_inventory.sql`
- CSV: `results/05_opioid_outcome/pyxis_medication_inventory.csv`
- Population: Frozen P4 adult LBP cohort

**Expected Output**

A concise, reproducible quality-assurance summary confirming whether:

- Medication name–GSN combinations are unique.
- Record and timing aggregates are internally consistent.
- Missing identifiers are quantified.
- Medication frequency distributions support a focused mapping approach.
- The inventory is suitable for proceeding to standardized Cumulus terminology matching.

No opioid membership decisions will be made in this section.

In [2]:

# 5.2.3 Source Data Quality & Mapping Readiness

inventory_path = (
    PROJECT_ROOT
    / "results"
    / "05_opioid_outcome"
    / "pyxis_medication_inventory.csv"
)

df_pyxis_inv = pd.read_csv(
    inventory_path,
    dtype={"pyxis_name": "string", "pyxis_gsn": "string"}
)

# --------------------------------------------------
# 1. Inventory integrity
# --------------------------------------------------

assert df_pyxis_inv[["pyxis_name", "pyxis_gsn"]].duplicated().sum() == 0
assert df_pyxis_inv["record_count"].gt(0).all()
assert df_pyxis_inv["encounter_count"].gt(0).all()
assert (
    df_pyxis_inv["encounter_count"]
    <= df_pyxis_inv["record_count"]
).all()

timing_columns = [
    "within_ed_records",
    "before_ed_records",
    "after_ed_records",
    "missing_charttime_records",
    "unclassified_time_records",
]

assert (
    df_pyxis_inv[timing_columns].sum(axis=1)
    == df_pyxis_inv["record_count"]
).all()

assert (
    df_pyxis_inv["missing_gsn_records"].gt(0)
    == df_pyxis_inv["pyxis_gsn"].isna()
).all()

# --------------------------------------------------
# 2. Mapping readiness metrics
# --------------------------------------------------

total_records = int(df_pyxis_inv["record_count"].sum())
unique_combinations = len(df_pyxis_inv)

top20_records = (
    df_pyxis_inv.nlargest(20, "record_count")
    ["record_count"].sum()
)

name_gsn_counts = (
    df_pyxis_inv.groupby("pyxis_name")
    ["pyxis_gsn"].nunique()
)

gsn_name_counts = (
    df_pyxis_inv.dropna(subset=["pyxis_gsn"])
    .groupby("pyxis_gsn")["pyxis_name"].nunique()
)

qa_summary = pd.DataFrame({
    "Metric": [
        "Total Pyxis records",
        "Unique name-GSN combinations",
        "Unique medication names",
        "Unique non-null GSN",
        "Missing-name records",
        "Missing-GSN records",
        "Missing-charttime records",
        "Duplicate name-GSN combinations",
        "Timing partition mismatches",
        "Names with multiple GSN",
        "GSN linked to multiple names",
        "Single-record combinations",
        "Top 20 combinations coverage (%)",
    ],
    "Observed": [
        total_records,
        unique_combinations,
        df_pyxis_inv["pyxis_name"].nunique(),
        df_pyxis_inv["pyxis_gsn"].nunique(),
        df_pyxis_inv["missing_name_records"].sum(),
        df_pyxis_inv["missing_gsn_records"].sum(),
        df_pyxis_inv["missing_charttime_records"].sum(),
        df_pyxis_inv[["pyxis_name", "pyxis_gsn"]]
            .duplicated().sum(),
        (
            df_pyxis_inv[timing_columns].sum(axis=1)
            != df_pyxis_inv["record_count"]
        ).sum(),
        (name_gsn_counts > 1).sum(),
        (gsn_name_counts > 1).sum(),
        (df_pyxis_inv["record_count"] == 1).sum(),
        round(top20_records / total_records * 100, 2),
    ],
})

display(qa_summary)

print("Pyxis inventory integrity: PASSED")


,Metric,Observed
0,Total Pyxis records,54814.00
1,Unique name-GSN combinations,1034.00
2,Unique medication names,712.00
3,Unique non-null GSN,518.00
4,Missing-name records,0.00
5,Missing-GSN records,334.00
6,Missing-charttime records,0.00
7,Duplicate name-GSN combinations,0.00
8,Timing partition mismatches,0.00
9,Names with multiple GSN,221.00


Pyxis inventory integrity: PASSED


#### Actual Results

The P4-linked Pyxis medication inventory passed all data-integrity checks.

| Validation Metric | Observed |
|---|---:|
| Total Pyxis records | 54,814 |
| Unique medication name–GSN combinations | 1,034 |
| Unique medication names | 712 |
| Unique non-null GSN identifiers | 518 |
| Missing medication names | 0 |
| Missing GSN records | 334 (0.61%) |
| Missing chart timestamps | 0 |
| Duplicate name–GSN combinations | 0 |
| Timing partition mismatches | 0 |
| Medication names linked to multiple GSN | 221 |
| GSN linked to multiple medication names | 388 |
| Single-record medication combinations | 122 |
| Top 20 medication combinations' record coverage | 48.17% |

**Validation Status: PASSED**

#### Findings

**1. Source inventory integrity**

The exported inventory is internally consistent, with unique medication name–GSN combinations, valid record frequencies, and complete reconciliation of medication timing categories.

No duplicate inventory keys or timing aggregation discrepancies were detected.

**2. Medication identifier variability**

Although medication names were complete, 334 source records lacked a GSN identifier. Multiple medication names were associated with different GSN identifiers, and individual GSN identifiers appeared alongside multiple names.

These findings support retaining both the original medication name and GSN during terminology mapping rather than assuming a one-to-one relationship.

**3. Mapping workload and prioritization**

The inventory contains 1,034 unique name–GSN combinations, with the 20 most frequent combinations accounting for 48.17% of source records.

This concentration supports a focused mapping approach that prioritizes high-frequency medications and records potentially relevant to opioid identification.

Low-frequency or unmatched records will not automatically be classified as nonopioid; potentially relevant records will receive targeted review.

**4. Readiness for standardized opioid identification**

The inventory provides sufficient source information to begin medication matching against the Cumulus.

Comprehensive normalization of all Pyxis medications is not required for the study objective. However, uncertain mappings that could affect encounter-level opioid labels must be assessed and documented.

#### Conclusion

The Pyxis medication inventory is validated and suitable for the focused standardized medication-mapping workflow.

Sections 5.2.1–5.2.3 establish the source-data evidence, cohort-linked medication inventory, and mapping-readiness assessment required for Section 5.3.

No additional medication inventory extraction or general Pyxis exploratory analysis is required at this stage.

## 5.3 Standardized Opioid Medication Mapping

### 5.3.1 Cumulus Reference Validation and Version Freeze

**Objective**

Validate the externally developed Cumulus Opioid Value Set and establish a reproducible reference for identifying opioid-related medications in MIMIC-IV-ED Pyxis records.

The published value set (Wasz et al., 2026) will serve as the medication classification reference. No independent clinical opioid inclusion or exclusion rules will be developed.

**Method**

1. Load the original Cumulus reference TSV from `codebooks/references/opioid_valueset.tsv`.
2. Validate its structure, RxCUI identifiers, missingness, and duplicate records.
3. Record the number of unique RxNorm concepts and associated terminology names.
4. Calculate a SHA-256 checksum to document file integrity.
5. Record the source repository, selected commit, and retrieval date separately for reproducibility.

The original TSV will remain unchanged.

**Expected Output**

A validated Cumulus reference containing RxNorm identifiers and medication terminology names, together with documented file-integrity metadata.

This reference will be used for focused Pyxis medication matching in Section 5.3.2.

The reference will not be treated as evidence of medication administration or clinical treatment indication.

In [3]:

# 5.3.1 Cumulus Reference Validation

import hashlib

assert CUMULUS_PATH.exists(), (
    f"Cumulus reference not found: {CUMULUS_PATH}"
)

# Original TSV has no header
df_cumulus = pd.read_csv(
    CUMULUS_PATH,
    sep="\t",
    header=None,
    names=["rxcui", "rxnorm_name"],
    dtype="string"
)

# Structural validation
assert df_cumulus.shape[1] == 2
assert df_cumulus.notna().all().all()
assert df_cumulus["rxcui"].str.fullmatch(r"\d+").all()
assert not df_cumulus.duplicated(
    subset=["rxcui", "rxnorm_name"]
).any()

# File integrity
sha256 = hashlib.sha256(
    CUMULUS_PATH.read_bytes()
).hexdigest()

reference_summary = pd.DataFrame({
    "Metric": [
        "Total terminology rows",
        "Unique RxCUI",
        "Unique medication names",
        "Missing RxCUI",
        "Missing names",
        "Duplicate RxCUI-name pairs",
    ],
    "Observed": [
        len(df_cumulus),
        df_cumulus["rxcui"].nunique(),
        df_cumulus["rxnorm_name"].nunique(),
        df_cumulus["rxcui"].isna().sum(),
        df_cumulus["rxnorm_name"].isna().sum(),
        df_cumulus.duplicated(
            ["rxcui", "rxnorm_name"]
        ).sum(),
    ],
})

display(reference_summary)

print(f"SHA-256: {sha256}")
print("Cumulus reference validation: PASSED")


,Metric,Observed
0,Total terminology rows,25743
1,Unique RxCUI,9225
2,Unique medication names,23961
3,Missing RxCUI,0
4,Missing names,0
5,Duplicate RxCUI-name pairs,0


SHA-256: 33eae9e60fa6584ed77e74bdf9a438dc8202eb161318af446d93d57cf55cf327
Cumulus reference validation: PASSED


#### Actual Results

The externally sourced Cumulus Opioid Value Set was successfully loaded and passed all structural and integrity checks.

| Validation Metric | Observed |
|---|---:|
| Total terminology rows | 25,743 |
| Unique RxCUI | 9,225 |
| Unique medication names | 23,961 |
| Missing RxCUI | 0 |
| Missing medication names | 0 |
| Duplicate RxCUI-name pairs | 0 |

**SHA-256:** `33eae9e60fa6584ed77e74bdf9a438dc8202eb161318af446d93d57cf55cf327`

**Validation Status: PASSED**

#### Findings

The reference file contains 9,225 distinct RxNorm concepts represented by 25,743 terminology rows. Multiple medication names may correspond to the same RxCUI, reflecting the terminology structure of the external reference.

All identifiers and medication names were present, and no duplicate RxCUI-name pairs were detected. The SHA-256 checksum confirms consistency with the previously inspected reference file.

The original TSV will remain unchanged in `codebooks/references/opioid_valueset.tsv`.

The value set provides an externally developed classification reference for subsequent Pyxis medication mapping. This structural validation does not establish mapping accuracy against local Pyxis medication names.

#### Conclusion

The Cumulus reference passed file-integrity validation and is ready for focused medication matching in Section 5.3.2.

The external Cumulus reference was preserved without modification. Its source repository, GitHub commit, retrieval date, and SHA-256 checksum were documented to support reproducibility.


### 5.3.2 Focused Pyxis-to-Cumulus Medication Matching

#### 5.3.2.1 Local Name Matching Pilot

**Objective**

Evaluate the feasibility of directly matching MIMIC-IV-ED Pyxis medication names to the frozen Cumulus Opioid Value Set without requiring an external terminology API.

This pilot will establish initial matching coverage, identify terminology variation, and estimate the scope of subsequent targeted review.

**Method**

1. Use the validated Pyxis medication inventory and Cumulus reference loaded in Sections 5.2.3 and 5.3.1.
2. Attempt case-insensitive exact-name matching.
3. Apply conservative text normalization to unmatched names, limited to whitespace and basic punctuation formatting.
4. Identify matches associated with multiple RxCUI identifiers.
5. Summarize matched and unmatched source medication combinations and record frequencies.
6. Inspect representative unmatched medication names to guide subsequent review.

Medication strengths, ingredients, combination components, and dosage forms will not be removed during normalization.

**Expected Output**

A preliminary medication-matching table containing the original Pyxis name and GSN, match status, matching method, candidate RxCUI identifiers, and source record frequencies.

The pilot will quantify direct terminology matching coverage without assuming that unmatched medications are nonopioid.

**Scope**

This is an exploratory terminology-matching step, not the final opioid classification. No custom clinical inclusion or exclusion rules will be introduced. Definitive encounter-level outcome labeling will occur only after the targeted mapping audit is completed.

In [4]:

# 5.3.2.1 Local Name Matching Pilot

import re
import unicodedata

# --------------------------------------------------
# 1. Conservative name normalization
# --------------------------------------------------

def normalize_med_name(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = text.casefold().strip()

    # Standardize whitespace around selected punctuation.
    # Do not remove ingredients, strengths, or dose forms.
    text = re.sub(r"\s*([/\-])\s*", r"\1", text)
    text = re.sub(r"\s+", " ", text)

    return text


pyxis = df_pyxis_inv.copy()
cumulus = df_cumulus.copy()

pyxis["exact_key"] = pyxis["pyxis_name"].str.casefold().str.strip()
cumulus["exact_key"] = cumulus["rxnorm_name"].str.casefold().str.strip()

pyxis["normalized_key"] = pyxis["pyxis_name"].map(normalize_med_name)
cumulus["normalized_key"] = cumulus["rxnorm_name"].map(normalize_med_name)

# --------------------------------------------------
# 2. Build local lookup dictionaries
# --------------------------------------------------

exact_lookup = (
    cumulus.groupby("exact_key")["rxcui"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

normalized_lookup = (
    cumulus.groupby("normalized_key")["rxcui"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

# --------------------------------------------------
# 3. Match Pyxis names
# --------------------------------------------------

def match_local(row):
    exact = exact_lookup.get(row["exact_key"], [])

    if exact:
        return pd.Series({
            "match_method": "exact",
            "candidate_rxcuis": exact,
        })

    normalized = normalized_lookup.get(
        row["normalized_key"], []
    )

    if normalized:
        return pd.Series({
            "match_method": "normalized",
            "candidate_rxcuis": normalized,
        })

    return pd.Series({
        "match_method": "unmatched",
        "candidate_rxcuis": [],
    })


match_results = pyxis.apply(match_local, axis=1)

df_mapping_pilot = pd.concat(
    [pyxis, match_results],
    axis=1
)

df_mapping_pilot["candidate_count"] = (
    df_mapping_pilot["candidate_rxcuis"].map(len)
)

df_mapping_pilot["match_status"] = np.select(
    [
        df_mapping_pilot["candidate_count"] > 1,
        df_mapping_pilot["candidate_count"] == 1,
    ],
    ["ambiguous", "matched"],
    default="unmatched",
)

# --------------------------------------------------
# 4. Summarize mapping coverage
# --------------------------------------------------

total_combinations = len(df_mapping_pilot)
total_records = df_mapping_pilot["record_count"].sum()

matched = df_mapping_pilot[
    df_mapping_pilot["match_status"] == "matched"
]

summary = (
    df_mapping_pilot
    .groupby(["match_method", "match_status"])
    .agg(
        combinations=("pyxis_name", "size"),
        source_records=("record_count", "sum")
    )
    .reset_index()
)

summary["combination_pct"] = (
    summary["combinations"] / total_combinations * 100
).round(2)

summary["record_pct"] = (
    summary["source_records"] / total_records * 100
).round(2)

display(summary)

print(f"Total combinations: {total_combinations:,}")
print(f"Total Pyxis records: {total_records:,}")
print(f"Unambiguous matched combinations: {len(matched):,}")
print(
    "Unambiguous matched record coverage: "
    f"{matched['record_count'].sum() / total_records:.2%}"
)

# --------------------------------------------------
# 5. Review representative matches / unmatched names
# --------------------------------------------------

print("\nTop matched combinations:")
display(
    df_mapping_pilot.query("match_status == 'matched'")
    .nlargest(15, "record_count")[
        ["pyxis_name", "pyxis_gsn", "record_count",
         "match_method", "candidate_rxcuis"]
    ]
)

print("\nTop unmatched combinations:")
display(
    df_mapping_pilot.query("match_status == 'unmatched'")
    .nlargest(25, "record_count")[
        ["pyxis_name", "pyxis_gsn", "record_count"]
    ]
)

print("\nAmbiguous combinations:")
display(
    df_mapping_pilot.query("match_status == 'ambiguous'")[
        ["pyxis_name", "pyxis_gsn", "record_count",
         "candidate_rxcuis"]
    ].head(20)
)


,match_method,match_status,combinations,source_records,combination_pct,record_pct
0,exact,matched,19,3443,1.84,6.28
1,unmatched,unmatched,1015,51371,98.16,93.72


Total combinations: 1,034
Total Pyxis records: 54,814
Unambiguous matched combinations: 19
Unambiguous matched record coverage: 6.28%

Top matched combinations:


,pyxis_name,pyxis_gsn,record_count,match_method,candidate_rxcuis
8,Morphine,004080,1308,exact,[7052]
21,Morphine Sulfate,004072,512,exact,[30236]
22,Morphine Sulfate,070024,512,exact,[30236]
30,Morphine Sulfate,074851,375,exact,[30236]
37,Morphine Sulfate,078456,310,exact,[30236]
75,Fentanyl Citrate,041385,93,exact,[142436]
76,Fentanyl Citrate,041384,93,exact,[142436]
101,Fentanyl,041385,69,exact,[4337]
143,Fentanyl,048287,52,exact,[4337]
268,Methadone,004240,25,exact,[6813]



Top unmatched combinations:


,pyxis_name,pyxis_gsn,record_count
0,Diazepam,003768,2494
1,Oxycodone-Acetaminophen,004222,2215
2,HYDROmorphone (Dilaudid),062823,2146
3,Acetaminophen,004490,2017
4,Acetaminophen 500mg TAB,004490,1868
5,Ketorolac,039500,1671
6,OxyCODONE (Immediate Rel 5mg TAB,004225,1496
7,OxyCODONE (Immediate Release),004225,1476
9,Ibuprofen,008349,1182
10,Morphine Sulfat 4mg/1mL 1mL VIAL,074851,1131



Ambiguous combinations:


,pyxis_name,pyxis_gsn,record_count,candidate_rxcuis


#### 5.3.2.2 Ingredient-Level Candidate Matching

**Objective**

Improve identification of potentially opioid-related Pyxis medications by addressing differences between local medication descriptions and standardized Cumulus terminology.

**Method**

1. Select a focused set of opioid-related ingredient names and verify that they appear in the frozen Cumulus reference.
2. Normalize Pyxis medication names using case-insensitive text matching.
3. Identify candidate ingredients within local medication descriptions, including combination products and entries containing strength or packaging information.
4. Retain the original medication name, GSN, matched ingredient names, and associated Cumulus RxCUI identifiers.
5. Summarize additional candidate matches and identify unresolved medications for targeted review.

The ingredient list is used only for candidate identification and does not replace or redefine the published Cumulus Opioid Value Set.

**Expected Output**

A preliminary ingredient-level matching summary showing the number of additional Pyxis medication combinations and source records identified, together with representative matched and unmatched medications.

Candidate matches will require verification before final outcome classification.

In [5]:

# 5.3.2.2 Ingredient-Level Candidate Matching

import re

# Common ingredient names used for candidate screening only.
ingredient_names = [
    "morphine", "hydromorphone", "oxycodone",
    "hydrocodone", "fentanyl", "tramadol",
    "codeine", "methadone", "buprenorphine",
    "meperidine", "oxymorphone", "tapentadol",
    "naloxone", "naltrexone", "nalbuphine",
    "butorphanol", "pentazocine",
    "sufentanil", "remifentanil", "alfentanil",
]

# Verify ingredient-name candidates against the frozen
# Cumulus reference. These names are not independently
# verified RxNorm TTY=IN/PIN classifications.
cumulus_name_lookup = (
    df_cumulus.assign(
        name_key=df_cumulus["rxnorm_name"]
        .str.casefold().str.strip()
    )
    .groupby("name_key")["rxcui"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

ingredient_reference = {
    name: cumulus_name_lookup[name]
    for name in ingredient_names
    if name in cumulus_name_lookup
}

missing_reference_names = sorted(
    set(ingredient_names) - set(ingredient_reference)
)

print("Ingredient names found in Cumulus:",
      len(ingredient_reference))
print("Names not found:", missing_reference_names)

# Use word boundaries to avoid partial-word false matches.
# This does not remove strength, salt, or packaging information.
patterns = {
    name: re.compile(
        rf"(?<![a-z]){re.escape(name)}(?![a-z])",
        re.IGNORECASE
    )
    for name in ingredient_reference
}

def identify_ingredients(med_name):
    if pd.isna(med_name):
        return []

    text = str(med_name)
    return [
        name for name, pattern in patterns.items()
        if pattern.search(text)
    ]

df_ingredient_pilot = df_mapping_pilot.copy()

df_ingredient_pilot["candidate_ingredients"] = (
    df_ingredient_pilot["pyxis_name"]
    .map(identify_ingredients)
)

df_ingredient_pilot["ingredient_rxcuis"] = (
    df_ingredient_pilot["candidate_ingredients"]
    .map(
        lambda names: sorted({
            rxcui
            for name in names
            for rxcui in ingredient_reference[name]
        })
    )
)

df_ingredient_pilot["ingredient_candidate"] = (
    df_ingredient_pilot["candidate_ingredients"]
    .map(bool)
)

# Newly identified candidates not found by exact matching.
new_candidates = df_ingredient_pilot[
    (df_ingredient_pilot["match_status"] == "unmatched")
    & df_ingredient_pilot["ingredient_candidate"]
].copy()

remaining = df_ingredient_pilot[
    (df_ingredient_pilot["match_status"] == "unmatched")
    & ~df_ingredient_pilot["ingredient_candidate"]
].copy()

print("\n--- Ingredient Matching Summary ---")
print(f"New candidate combinations: {len(new_candidates):,}")
print(
    "Source records in new candidates: "
    f"{new_candidates['record_count'].sum():,}"
)
print(f"Remaining unmatched combinations: {len(remaining):,}")
print(
    "Remaining unmatched source records: "
    f"{remaining['record_count'].sum():,}"
)

print("\n--- Top Newly Identified Candidates ---")
display(
    new_candidates.nlargest(25, "record_count")[
        ["pyxis_name", "pyxis_gsn", "record_count",
         "candidate_ingredients", "ingredient_rxcuis"]
    ]
)

print("\n--- Top Remaining Unmatched ---")
display(
    remaining.nlargest(25, "record_count")[
        ["pyxis_name", "pyxis_gsn", "record_count"]
    ]
)


Ingredient names found in Cumulus: 19
Names not found: ['codeine']

--- Ingredient Matching Summary ---
New candidate combinations: 92
Source records in new candidates: 12,360
Remaining unmatched combinations: 923
Remaining unmatched source records: 39,011

--- Top Newly Identified Candidates ---


,pyxis_name,pyxis_gsn,record_count,candidate_ingredients,ingredient_rxcuis
1,Oxycodone-Acetaminophen,004222,2215,[oxycodone],[7804]
2,HYDROmorphone (Dilaudid),062823,2146,[hydromorphone],[3423]
6,OxyCODONE (Immediate Rel 5mg TAB,004225,1496,[oxycodone],[7804]
7,OxyCODONE (Immediate Release),004225,1476,[oxycodone],[7804]
10,Morphine Sulfat 4mg/1mL 1mL VIAL,074851,1131,[morphine],[7052]
14,HYDROmorphone (D 1mg/1mL 1mL SYR,062823,826,[hydromorphone],[3423]
35,HYDROmorphone (Dilaudid),004110,335,[hydromorphone],[3423]
36,OxycoDONE (Immediate Release),004225,331,[oxycodone],[7804]
42,Oxycodone-Acetaminophen 1TAB TAB,004222,256,[oxycodone],[7804]
45,HYDROmorphone (Dilaudid),078729,234,[hydromorphone],[3423]



--- Top Remaining Unmatched ---


,pyxis_name,pyxis_gsn,record_count
0,Diazepam,003768,2494
3,Acetaminophen,004490,2017
4,Acetaminophen 500mg TAB,004490,1868
5,Ketorolac,039500,1671
9,Ibuprofen,008349,1182
11,Lidocaine 5% Patch 1PTCH PTCH,043256,1093
12,Ketorolac 30mg/1mL 1mL VIAL,039500,915
13,Diazepam 5mg TAB,003768,838
15,Lidocaine 5% Patch,043256,796
16,Ondansetron,015869,788


#### Actual Results

Ingredient-level candidate matching identified 92 additional Pyxis medication name–GSN combinations, representing 12,360 source records that had not matched the Cumulus reference through direct name matching.

| Matching Category | Combinations | Source Records |
|---|---:|---:|
| Direct Cumulus name matches | 19 | 3,443 |
| Additional ingredient candidates | 92 | 12,360 |
| Remaining unmatched | 923 | 39,011 |
| **Total** | **1,034** | **54,814** |

Together, direct and ingredient-level matching identified 111 candidate medication combinations associated with 15,803 Pyxis source records (28.83%).

Of the 20 predefined screening names, 19 were found as exact medication names in the Cumulus reference. The standalone name `codeine` was not found and requires additional inspection.

#### Findings

**1. Improved identification of locally formatted medications**

Ingredient-level matching recovered high-frequency opioid candidates that were missed by direct name matching, including oxycodone/acetaminophen combinations, hydromorphone products, strength-specific morphine preparations, and tramadol formulations.

**2. Remaining unmatched records**

The highest-frequency remaining unmatched medications were predominantly recognizable nonopioid medications, including acetaminophen, ketorolac, ibuprofen, diazepam, and lidocaine.

However, the remaining unmatched records have not been fully reviewed and cannot yet be classified collectively as nonopioid.

**3. Terminology limitations**

Ingredient screening identifies candidate opioid-related medications without establishing an exact RxNorm product-level mapping. Some brand-only, abbreviated, or truncated medication names may remain unresolved.

#### Conclusion

Ingredient-level candidate matching substantially improved the identification of potentially relevant opioid medications using the frozen Cumulus reference.

The next step is a focused audit of remaining opioid candidates, particularly codeine-containing medications and locally abbreviated product names, before finalizing the medication mapping dictionary.

In [6]:

# 5.3.2.3 Preliminary Unmatched Candidate Audit

# 1. Check codeine-related terminology in Cumulus
print("--- Codeine-related Cumulus terminology ---")
codeine_reference = df_cumulus[
    df_cumulus["rxnorm_name"].str.contains(
        "codein", case=False, na=False
    )
]

print("Cumulus codeine-related rows:",
      len(codeine_reference))
display(codeine_reference.head(20))

# 2. Check codeine-related Pyxis names
print("\n--- Codeine-related Pyxis medications ---")
codeine_pyxis = df_ingredient_pilot[
    df_ingredient_pilot["pyxis_name"].str.contains(
        "codein", case=False, na=False
    )
]

display(codeine_pyxis[
    ["pyxis_name", "pyxis_gsn", "record_count",
     "match_method", "candidate_ingredients"]
])

# 3. Identify remaining opioid-like names for review.
# Screening patterns only; not classification rules.
audit_pattern = (
    r"codein|morph|codon|fentan|tramad|methadon|"
    r"buprenorph|meperidin|tapentadol|nalox|"
    r"nalbuphin|butorphan|pentazocin|"
    r"propoxyphen|levorphan|sufentan|remifentan|"
    r"alfentan|dilaudid|percocet|vicodin|"
    r"norco|ultram|suboxone|oxycontin"
)

audit_candidates = remaining[
    remaining["pyxis_name"].str.contains(
        audit_pattern, case=False, na=False, regex=True
    )
].copy()

print("\n--- Remaining opioid-like candidates ---")
print("Candidate combinations:", len(audit_candidates))
print("Associated source records:",
      audit_candidates["record_count"].sum())

display(
    audit_candidates.sort_values(
        "record_count", ascending=False
    )[
        ["pyxis_name", "pyxis_gsn", "record_count"]
    ]
)


--- Codeine-related Cumulus terminology ---
Cumulus codeine-related rows: 5123


,rxcui,rxnorm_name
3,1007204,codeine / propyphenazone
4,1007238,aconite / codeine / Erysimum preparation
6,1007293,codeine / potassium
7,1007354,chlorpheniramine / dihydrocodeine / phenylephr...
8,1007354,chlorpheniramine/dihydrocodeine/PE/PPA
10,1007477,codeine / ethylmorphine
12,1007568,codeine / guaiacolsulfonic acid / phenylephrin...
13,1007569,acetaminophen / chlorpheniramine / codeine
14,1007569,acetaminophen/chlorpheniramine/codeine
15,1007572,acetaminophen / codeine / guaifenesin / pseudo...



--- Codeine-related Pyxis medications ---


,pyxis_name,pyxis_gsn,record_count,match_method,candidate_ingredients
195,Acetaminophen w/Codeine,004165,34,unmatched,[]
486,Acetaminophen w/Codeine 1TAB TAB,004165,10,unmatched,[]
938,Codeine,004186,1,unmatched,[]



--- Remaining opioid-like candidates ---
Candidate combinations: 3
Associated source records: 45


,pyxis_name,pyxis_gsn,record_count
195,Acetaminophen w/Codeine,004165,34
486,Acetaminophen w/Codeine 1TAB TAB,004165,10
938,Codeine,004186,1


#### Actual Results

A targeted keyword audit was conducted on the 923 Pyxis name–GSN combinations remaining unmatched after direct and ingredient-level matching.

The audit identified three additional codeine-related combinations, representing 45 Pyxis source records.

| Medication | GSN | Source Records |
|---|---|---:|
| Acetaminophen w/Codeine | 004165 | 34 |
| Acetaminophen w/Codeine 1TAB TAB | 004165 | 10 |
| Codeine | 004186 | 1 |
| **Total** | | **45** |

The Cumulus reference contained 5,123 terminology rows with names containing the substring `codein`, including codeine-containing combination products. However, the standalone name `codeine` was not found through the earlier exact-name lookup.

Across all screening stages, 114 Pyxis name–GSN combinations representing 15,848 source records were identified as potential Cumulus-related medications.

#### Findings

Ingredient-level screening captured most high-frequency, recognizable opioid candidates, while the targeted keyword audit identified an additional group of codeine-related medications.

The remaining 920 unflagged combinations were not individually validated. Therefore, the audit cannot establish complete opioid identification or exclude the possibility of missed brand-only, abbreviated, or truncated medication names.

The 114 candidate combinations require a final classification review before encounter-level outcome construction.

#### Conclusion

The targeted keyword audit identified a small additional group of potentially relevant codeine medications and supports proceeding to a focused mapping review.

The frozen Cumulus reference remains the medication classification standard. Keyword matches are treated as screening evidence rather than independent confirmation of RxNorm membership.

In [7]:

# 5.3.3.1 GSN Consistency Check

# Combine candidate status from previous steps
df_audit = df_ingredient_pilot.copy()

codeine_indices = audit_candidates.index

df_audit["opioid_candidate"] = (
    (df_audit["match_status"] != "unmatched")
    | df_audit["ingredient_candidate"]
    | df_audit.index.isin(codeine_indices)
)

# Use only nonmissing GSN identifiers
known = df_audit[
    df_audit["pyxis_gsn"].notna()
].copy()

# GSNs already associated with candidate medications
candidate_gsns = set(
    known.loc[
        known["opioid_candidate"], "pyxis_gsn"
    ]
)

# Previously unflagged names sharing those GSNs
gsn_review = known[
    ~known["opioid_candidate"]
    & known["pyxis_gsn"].isin(candidate_gsns)
].copy()

print("Unflagged combinations sharing candidate GSNs:",
      len(gsn_review))

print("Associated source records:",
      gsn_review["record_count"].sum())

display(
    gsn_review.sort_values(
        "record_count", ascending=False
    )[
        ["pyxis_name", "pyxis_gsn", "record_count"]
    ]
)


Unflagged combinations sharing candidate GSNs: 1
Associated source records: 3


,pyxis_name,pyxis_gsn,record_count
768,LOPERamide 2mg CAP,002842,3


#### Actual Results

The GSN consistency check identified one previously unflagged Pyxis medication combination sharing a GSN with an existing opioid-related candidate.

| Pyxis Medication | GSN | Source Records |
|---|---|---:|
| LOPERamide 2mg CAP | 002842 | 3 |

The frozen Cumulus reference includes both the ingredient-level term `loperamide` (RxCUI 6468) and a corresponding oral capsule concept (RxCUI 978006).

Including this additional candidate brings the screening inventory to 115 medication name–GSN combinations representing 15,851 Pyxis source records.

#### Findings

The additional match illustrates the broad clinical scope of the Cumulus reference. Loperamide is an opioid-receptor-active antidiarrheal medication rather than a conventional opioid analgesic for low back pain.

The GSN consistency check provides supporting evidence for medication identification but does not independently establish product-level RxNorm equivalence.

#### Conclusion

The GSN review identified one additional relevant medication candidate. The mapping workflow will retain the frozen Cumulus terminology definition and explicitly describe its broad clinical scope during outcome validation.

Final classification and the assessment of potentially missed medications remain necessary before encounter-level outcome construction.

### 5.3.3.2 Final Candidate Review and Residual Risk Audit

**Objective**

Consolidate opioid-related medication candidates identified through direct Cumulus matching, ingredient-level screening, keyword review, and GSN consistency checks. Assess the remaining unmatched medication inventory for potential false-negative classifications.

**Method**

1. Combine the four candidate-identification methods into one non-overlapping review inventory.
2. Preserve original Pyxis medication names, GSN identifiers, record frequencies, candidate ingredients, and RxCUI evidence.
3. Review candidate medications before assigning final opioid classification.
4. Aggregate the remaining unflagged combinations by medication name to reduce redundant review.
5. Prioritize high-frequency unresolved names and inspect the remaining unique names for potential missed opioid-related medications.

**Expected Output**

A consolidated candidate inventory and a residual unmatched medication inventory, with source record counts and evidence available for targeted manual review.

**Decision Boundary**

Candidate identification does not establish final medication classification. Unflagged medications will not automatically be considered verified nonopioids. Any unresolved medication classifications will be documented before constructing encounter-level outcome labels.

In [8]:

# 5.3.3.2 Consolidated Candidate Review

df_review = df_ingredient_pilot.copy()

# Each combination receives one primary screening category.
df_review["screening_method"] = np.select(
    [
        df_review["match_status"] != "unmatched",
        df_review["ingredient_candidate"],
        df_review.index.isin(audit_candidates.index),
        df_review.index.isin(gsn_review.index),
    ],
    [
        "direct_cumulus",
        "ingredient_candidate",
        "keyword_candidate",
        "gsn_candidate",
    ],
    default="unflagged",
)

df_review["needs_review"] = (
    df_review["screening_method"] != "unflagged"
)

screening_summary = (
    df_review.groupby("screening_method")
    .agg(
        combinations=("pyxis_name", "size"),
        source_records=("record_count", "sum"),
        unique_names=("pyxis_name", "nunique"),
    )
    .reset_index()
)

display(screening_summary)

assert len(df_review) == 1034
assert df_review["record_count"].sum() == 54814
assert df_review["needs_review"].sum() == 115

# Candidate medications
df_candidate_review = (
    df_review[df_review["needs_review"]]
    .sort_values("record_count", ascending=False)
    .copy()
)

print("\n--- Candidate Medication Review ---")
print("Combinations:", len(df_candidate_review))
print(
    "Source records:",
    df_candidate_review["record_count"].sum()
)

display(
    df_candidate_review[
        [
            "pyxis_name",
            "pyxis_gsn",
            "record_count",
            "screening_method",
            "candidate_ingredients",
            "candidate_rxcuis",
            "ingredient_rxcuis",
        ]
    ]
)

# Remaining unflagged combinations
df_unflagged = df_review[
    ~df_review["needs_review"]
].copy()

# Aggregate repeated combinations to unique medication names.
df_unflagged_names = (
    df_unflagged.groupby("pyxis_name")
    .agg(
        combinations=("pyxis_gsn", "size"),
        source_records=("record_count", "sum"),
        unique_gsns=("pyxis_gsn", "nunique"),
    )
    .sort_values("source_records", ascending=False)
    .reset_index()
)

print("\n--- Residual Unflagged Inventory ---")
print("Combinations:", len(df_unflagged))
print("Unique names:", len(df_unflagged_names))
print("Source records:", df_unflagged["record_count"].sum())

display(df_unflagged_names.head(50))

# Save local review worklists; not final classifications.
REVIEW_DIR = PROJECT_ROOT / "results" / "05_opioid_outcome"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

df_candidate_review.to_csv(
    REVIEW_DIR / "opioid_candidate_review.csv",
    index=False
)

df_unflagged_names.to_csv(
    REVIEW_DIR / "unflagged_medication_review.csv",
    index=False
)

print("\nReview worklists exported.")


,screening_method,combinations,source_records,unique_names
0,direct_cumulus,19,3443,7
1,gsn_candidate,1,3,1
2,ingredient_candidate,92,12360,61
3,keyword_candidate,3,45,3
4,unflagged,919,38963,640



--- Candidate Medication Review ---
Combinations: 115
Source records: 15851


,pyxis_name,pyxis_gsn,record_count,screening_method,candidate_ingredients,candidate_rxcuis,ingredient_rxcuis
1,Oxycodone-Acetaminophen,004222,2215,ingredient_candidate,[oxycodone],[],[7804]
2,HYDROmorphone (Dilaudid),062823,2146,ingredient_candidate,[hydromorphone],[],[3423]
6,OxyCODONE (Immediate Rel 5mg TAB,004225,1496,ingredient_candidate,[oxycodone],[],[7804]
7,OxyCODONE (Immediate Release),004225,1476,ingredient_candidate,[oxycodone],[],[7804]
8,Morphine,004080,1308,direct_cumulus,[morphine],[7052],[7052]
...,...,...,...,...,...,...,...
951,Fentanyl Patch 100mcg/h PTC,015883,1,ingredient_candidate,[fentanyl],[],[4337]
950,Fentanyl Patch,015880,1,ingredient_candidate,[fentanyl],[],[4337]
938,Codeine,004186,1,keyword_candidate,[],[],[]
923,Buprenorphine-Naloxone 1TAB TAB,051641,1,ingredient_candidate,"[buprenorphine, naloxone]",[],"[1819, 7242]"



--- Residual Unflagged Inventory ---
Combinations: 919
Unique names: 640
Source records: 38963


,pyxis_name,combinations,source_records,unique_gsns
0,Diazepam,6,2865,6
1,Acetaminophen,2,2658,2
2,Ketorolac,3,2033,3
3,Ibuprofen,2,1875,2
4,Acetaminophen 500mg TAB,1,1868,1
5,Ondansetron,2,1576,2
6,Lorazepam,4,1561,4
7,Lidocaine 5% Patch 1PTCH PTCH,1,1093,1
8,Ketorolac 30mg/1mL 1mL VIAL,1,915,1
9,LORazepam 2mg/1mL 1mL SYR,3,855,3



Review worklists exported.


#### Actual Results

The consolidated screening inventory contained 1,034 unique Pyxis medication name–GSN combinations representing 54,814 source records.

| Screening Category | Combinations | Source Records |
|---|---:|---:|
| Direct Cumulus match | 19 | 3,443 |
| Ingredient-level candidate | 92 | 12,360 |
| Keyword-based candidate | 3 | 45 |
| GSN-linked candidate | 1 | 3 |
| Unflagged | 919 | 38,963 |
| **Total** | **1,034** | **54,814** |

The 115 opioid-related candidates accounted for 15,851 source records (28.92%).

The remaining 919 unflagged combinations represented 640 unique medication names. Their full name-level inventory was inspected for potentially missed opioid-related ingredients, brand names, abbreviations, and truncated medication descriptions.

#### Findings

**1. Candidate identification**

Direct Cumulus matching, ingredient-level screening, targeted codeine review, and GSN consistency checks identified 115 medication combinations requiring final classification.

**2. Residual medication review**

The 640 unflagged medication names were predominantly recognizable nonopioid medications, including nonopioid analgesics, benzodiazepines, muscle relaxants, antiemetics, antibiotics, and other commonly used medications.

No additional clearly identifiable opioid-related medication names were found during review of the residual inventory.

**3. Residual uncertainty**

The review was based on available local medication descriptions and did not independently verify every unflagged combination against the complete RxNorm terminology system.

Therefore, the absence of additional recognizable candidates does not establish perfect sensitivity or eliminate all potential medication misclassification.

#### Conclusion

The candidate-screening and residual-name review stages are complete. The results support proceeding to construction of a reproducible medication mapping dictionary.

Final classification will distinguish confirmed Cumulus-related medications from reviewed nonopioid medications and any unresolved cases. Screening results will not be treated as verified encounter-level dispensing outcomes until classification and temporal attribution are completed.

### 5.3.4 Final Medication Mapping Dictionary

#### 5.3.4.1 Prepare Medication Mapping Dictionary

**Objective**

Consolidate the medication matching and audit results into a reproducible Pyxis medication mapping dictionary for subsequent opioid dispensing outcome construction.

**Method**

1. Retain the original Pyxis medication name and GSN for each unique medication combination.
2. Consolidate direct Cumulus matches, ingredient-level candidates, keyword-identified candidates, and GSN-linked candidates.
3. Assign preliminary classification values while preserving the distinction between terminology evidence and final reviewed classifications.
4. Retain the original Cumulus RxCUI evidence where available.
5. Export a draft mapping dictionary for final validation.

The published Cumulus reference will remain unchanged. Ingredient-level matching is used to identify opioid-containing medications but will not be represented as exact product-level RxNorm mapping.

**Expected Output**

A draft medication mapping dictionary containing all 1,034 Pyxis name–GSN combinations, preliminary opioid classifications, mapping evidence, and review status.

**Validation Boundary**

The draft will not be used to construct encounter-level outcomes until the remaining candidate classifications are reviewed and the final mapping dictionary passes integrity checks.

In [9]:

# 5.3.4.1 Prepare Medication Mapping Dictionary

df_mapping = df_review.copy()

# Preliminary classification based on prior screening
df_mapping["proposed_opioid"] = np.where(
    df_mapping["screening_method"] == "unflagged",
    0,
    1
)

# Preserve evidence rather than treating ingredient hits
# as verified product-level RxNorm mappings.
def collect_rxcuis(row):
    direct = row["candidate_rxcuis"]
    ingredients = row["ingredient_rxcuis"]

    return sorted(set(direct) | set(ingredients))

df_mapping["evidence_rxcuis"] = df_mapping.apply(
    collect_rxcuis, axis=1
)

df_mapping["evidence_type"] = df_mapping[
    "screening_method"
].map({
    "direct_cumulus": "direct_name_match",
    "ingredient_candidate": "ingredient_name_match",
    "keyword_candidate": "keyword_screening",
    "gsn_candidate": "gsn_linked_candidate",
    "unflagged": "residual_name_screening",
})

# Candidates must be confirmed before freezing.
df_mapping["review_status"] = np.where(
    df_mapping["proposed_opioid"] == 1,
    "pending_candidate_review",
    "name_screened_nonopioid"
)

# Final classification remains unset until validation.
df_mapping["is_cumulus_opioid"] = pd.Series(
    pd.array([pd.NA] * len(df_mapping), dtype="Int64"),
    index=df_mapping.index
)

mapping_columns = [
    "pyxis_name",
    "pyxis_gsn",
    "record_count",
    "screening_method",
    "candidate_ingredients",
    "evidence_rxcuis",
    "evidence_type",
    "proposed_opioid",
    "review_status",
    "is_cumulus_opioid",
]

df_mapping = df_mapping[mapping_columns].copy()

# Integrity checks
assert len(df_mapping) == 1034
assert df_mapping["record_count"].sum() == 54814

assert not df_mapping.duplicated(
    ["pyxis_name", "pyxis_gsn"]
).any()

assert df_mapping["proposed_opioid"].sum() == 115
assert df_mapping["is_cumulus_opioid"].isna().all()

# Summary
mapping_summary = (
    df_mapping.groupby(
        ["proposed_opioid", "review_status"]
    )
    .agg(
        combinations=("pyxis_name", "size"),
        source_records=("record_count", "sum")
    )
    .reset_index()
)

display(mapping_summary)

print("\nCandidate evidence summary:")
display(
    df_mapping[
        df_mapping["proposed_opioid"] == 1
    ].groupby("evidence_type")
    .agg(
        combinations=("pyxis_name", "size"),
        source_records=("record_count", "sum")
    )
)

# Export draft locally, not to the final codebooks folder
DRAFT_PATH = (
    PROJECT_ROOT
    / "results"
    / "05_opioid_outcome"
    / "opioid_medication_mapping_draft.csv"
)

df_mapping.to_csv(DRAFT_PATH, index=False)

print(f"\nDraft mapping saved: {DRAFT_PATH}")
print("Draft mapping integrity checks: PASSED")


,proposed_opioid,review_status,combinations,source_records
0,0,name_screened_nonopioid,919,38963
1,1,pending_candidate_review,115,15851



Candidate evidence summary:


,combinations,source_records
evidence_type,,
direct_name_match,19,3443
gsn_linked_candidate,1,3
ingredient_name_match,92,12360
keyword_screening,3,45



Draft mapping saved: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/05_opioid_outcome/opioid_medication_mapping_draft.csv
Draft mapping integrity checks: PASSED


#### Actual Results

The preliminary medication mapping dictionary was successfully generated from the consolidated Pyxis screening inventory.

| Classification | Combinations | Source Records |
|---|---:|---:|
| Proposed opioid-related | 115 | 15,851 |
| Name-screened nonopioid | 919 | 38,963 |
| **Total** | **1,034** | **54,814** |

The 115 opioid-related candidates were identified through direct Cumulus matching (19 combinations), ingredient-level screening (92), keyword screening (3), and GSN-based review (1).

All 1,034 original Pyxis medication name–GSN combinations were retained without duplicate keys. Source record counts reconciled to the previously validated total of 54,814 records.

**Integrity Checks: PASSED**

#### Findings

The draft dictionary preserves the original Pyxis identifiers, candidate classification, terminology evidence, and review status.

Final `is_cumulus_opioid` values remain unassigned to distinguish preliminary screening decisions from reviewed medication classifications.

The draft was exported to `results/05_opioid_outcome/opioid_medication_mapping_draft.csv`.

#### Conclusion

The draft mapping dictionary is structurally complete and ready for final medication classification review. It will not be used for encounter-level outcome construction until the classification review and final integrity checks are completed.

#### 5.3.4.2 Final Mapping Evidence Review

**Objective**

Complete the final evidence review of the draft Pyxis medication mapping dictionary before assigning confirmed classifications and freezing the mapping file.

**Method**

1. Verify that all 1,034 Pyxis medication combinations are retained without duplicate keys.
2. Review Cumulus terminology evidence for the three codeine-related candidates.
3. Verify the GSN-linked loperamide candidate against the frozen Cumulus reference.
4. Identify candidate records with missing GSN identifiers to inform subsequent SQL joins.
5. Preserve the distinction between exact RxNorm matches, ingredient-level evidence, and name-based screening.

**Expected Output**

A final evidence summary identifying any unresolved medication classifications or technical issues requiring attention before the mapping dictionary is frozen.

No medication classification will be assigned solely on the basis of a keyword hit or shared GSN.

In [10]:

# 5.3.4.2 Final Mapping Evidence Review

# 1. Verify the draft inventory
assert len(df_mapping) == 1034
assert df_mapping["record_count"].sum() == 54814
assert not df_mapping.duplicated(
    ["pyxis_name", "pyxis_gsn"]
).any()

cumulus_ids = set(df_cumulus["rxcui"].astype(str))

# 2. Check Codeine reference evidence
print("--- Codeine Reference Check ---")

codeine_ids = ["2670", "2672", "91104"]

display(
    df_cumulus[
        df_cumulus["rxcui"].isin(codeine_ids)
    ]
)

print("Reference ingredient/salt IDs present:",
      sorted(cumulus_ids.intersection(codeine_ids)))

codeine_products = df_cumulus[
    df_cumulus["rxnorm_name"].str.contains(
        r"(?<![a-z])codeine(?![a-z])",
        case=False, na=False, regex=True
    )
]

print("Codeine-containing terminology rows:",
      len(codeine_products))
display(codeine_products.head(10))

# 3. Review the three local Codeine candidates
print("\n--- Pyxis Codeine Candidates ---")

display(
    df_mapping[
        df_mapping["screening_method"] == "keyword_candidate"
    ][
        ["pyxis_name", "pyxis_gsn",
         "record_count", "evidence_rxcuis"]
    ]
)

# 4. Verify Loperamide reference
print("\n--- Loperamide Reference Check ---")

display(
    df_cumulus[
        df_cumulus["rxcui"].isin(
            ["6468", "978006"]
        )
    ]
)

# 5. Identify missing-GSN records
print("\n--- Missing GSN Check ---")

missing_gsn = df_mapping[
    df_mapping["pyxis_gsn"].isna()
    | (df_mapping["pyxis_gsn"].astype(str).str.strip() == "")
]

print("Missing-GSN combinations:", len(missing_gsn))
print("Missing-GSN candidate combinations:",
      missing_gsn["proposed_opioid"].sum())

display(
    missing_gsn[
        missing_gsn["proposed_opioid"] == 1
    ][
        ["pyxis_name", "pyxis_gsn",
         "record_count", "screening_method"]
    ]
)

print("\nFinal evidence inspection completed.")


--- Codeine Reference Check ---


,rxcui,rxnorm_name
13758,2670,(-)-Codeine
13759,2670,"(5alpha,6alpha)-7,8-Didehydro-4,5-epoxy-3-meth..."
13760,2670,"7,8-didehydro-4,5α-epoxy-3-methoxy-17-methylmo..."
13761,2670,Codein
13762,2670,Codeína
13763,2670,Codéine
13764,2670,Codeine (substance)
13765,2670,Codeine anhydrous
13766,2670,Codeine-containing product
13767,2670,Codeinum


Reference ingredient/salt IDs present: ['2670', '2672', '91104']
Codeine-containing terminology rows: 4414


,rxcui,rxnorm_name
3,1007204,codeine / propyphenazone
4,1007238,aconite / codeine / Erysimum preparation
6,1007293,codeine / potassium
10,1007477,codeine / ethylmorphine
12,1007568,codeine / guaiacolsulfonic acid / phenylephrin...
13,1007569,acetaminophen / chlorpheniramine / codeine
14,1007569,acetaminophen/chlorpheniramine/codeine
15,1007572,acetaminophen / codeine / guaifenesin / pseudo...
16,1007572,acetaminophen/codeine/guaifenesin/PSE
23,1008059,brompheniramine / codeine / guaifenesin



--- Pyxis Codeine Candidates ---


,pyxis_name,pyxis_gsn,record_count,evidence_rxcuis
195,Acetaminophen w/Codeine,004165,34,[]
486,Acetaminophen w/Codeine 1TAB TAB,004165,10,[]
938,Codeine,004186,1,[]



--- Loperamide Reference Check ---


,rxcui,rxnorm_name
15414,6468,Loperamid
15415,6468,Loperamida
15416,6468,loperamide
15417,6468,Loperamide (substance)
15418,6468,Loperamide-containing product
15419,6468,Loperamidum
15420,6468,Product containing loperamide (medicinal product)
22992,978006,"Anti-Diarrheal, 2 mg oral capsule"
22993,978006,CVS Anti-Diarrheal 2mg Softgel
22994,978006,Equate Anti-Diarrheal 2mg Softgel



--- Missing GSN Check ---
Missing-GSN combinations: 28
Missing-GSN candidate combinations: 1


,pyxis_name,pyxis_gsn,record_count,screening_method
760,Fentanyl (Pharmac 2.5mg 50mL Bag,<NA>,3,ingredient_candidate



Final evidence inspection completed.


#### Actual Results

The final terminology evidence review confirmed Cumulus membership for the previously unresolved codeine and loperamide candidates.

| Validation Item | Observed |
|---|---|
| Codeine ingredient RxCUI | 2670 — present |
| Codeine phosphate RxCUI | 2672 — present |
| Codeine sulfate RxCUI | 91104 — present |
| Loperamide ingredient RxCUI | 6468 — present |
| Loperamide 2 mg oral capsule RxCUI | 978006 — present |
| Medication combinations with missing GSN | 28 |
| Opioid-related candidates with missing GSN | 1 |

The three codeine-related combinations represented 45 source records. The GSN-linked loperamide combination represented three source records. Both were supported by terminology concepts present in the frozen Cumulus reference.

One fentanyl-related candidate, representing three source records, had a missing GSN but retained ingredient-level evidence.

#### Findings

The final evidence check resolved the previously identified codeine and loperamide terminology gaps without requiring an external terminology API.

Missing GSN values do not prevent name-based medication classification but require null-safe handling during subsequent SQL integration.

Ingredient-level RxCUI evidence supports opioid-related medication identification but does not establish exact RxNorm product-level equivalence.

#### Conclusion

The medication mapping evidence review is complete. The reviewed mapping decisions can now be frozen for outcome construction, with residual uncertainty documented for medications classified through local name screening.


#### 5.3.4.3 Freeze Final Medication Mapping Dictionary

The reviewed medication classifications are finalized and exported as a versioned mapping dictionary. The original Pyxis identifiers and Cumulus terminology evidence are preserved for reproducibility.

The exported dictionary is reloaded and checked for completeness, unique medication keys, and consistency with the validated medication inventory.


In [11]:

# 5.3.4.3 Freeze Final Medication Mapping Dictionary

import ast

df_final_mapping = df_mapping.copy()

# Convert serialized list fields into Python lists.
def parse_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or str(value).strip() == "":
        return []
    return ast.literal_eval(str(value))

df_final_mapping["evidence_rxcuis"] = (
    df_final_mapping["evidence_rxcuis"].map(parse_list)
)

df_final_mapping["candidate_ingredients"] = (
    df_final_mapping["candidate_ingredients"].map(parse_list)
)

# --------------------------------------------------
# 1. Resolve previously reviewed terminology gaps
# --------------------------------------------------

codeine_mask = (
    df_final_mapping["screening_method"] == "keyword_candidate"
) & (
    df_final_mapping["pyxis_name"].str.contains(
        "codeine", case=False, na=False
    )
)

loperamide_mask = (
    df_final_mapping["screening_method"] == "gsn_candidate"
) & (
    df_final_mapping["pyxis_name"].str.casefold()
    == "loperamide 2mg cap"
)

assert codeine_mask.sum() == 3
assert loperamide_mask.sum() == 1

df_final_mapping.loc[
    codeine_mask, "candidate_ingredients"
] = pd.Series(
    [["codeine"]] * 3,
    index=df_final_mapping.index[codeine_mask]
)

df_final_mapping.loc[
    codeine_mask, "evidence_rxcuis"
] = pd.Series(
    [["2670"]] * 3,
    index=df_final_mapping.index[codeine_mask]
)

df_final_mapping.loc[
    loperamide_mask, "candidate_ingredients"
] = pd.Series(
    [["loperamide"]],
    index=df_final_mapping.index[loperamide_mask]
)

df_final_mapping.loc[
    loperamide_mask, "evidence_rxcuis"
] = pd.Series(
    [["6468", "978006"]],
    index=df_final_mapping.index[loperamide_mask]
)

# --------------------------------------------------
# 2. Verify all evidence RxCUIs against Cumulus
# --------------------------------------------------

cumulus_ids = set(df_cumulus["rxcui"].astype(str))

all_evidence_ids = {
    str(rxcui)
    for values in df_final_mapping["evidence_rxcuis"]
    for rxcui in values
}

assert all_evidence_ids.issubset(cumulus_ids), (
    "Some evidence RxCUIs are absent from Cumulus."
)

candidate_mask = df_final_mapping["proposed_opioid"] == 1

assert df_final_mapping.loc[
    candidate_mask, "evidence_rxcuis"
].map(bool).all(), (
    "Some opioid candidates lack Cumulus evidence."
)

# --------------------------------------------------
# 3. Freeze reviewed operational classifications
# --------------------------------------------------

df_final_mapping["is_cumulus_opioid"] = (
    df_final_mapping["proposed_opioid"].astype("Int64")
)

df_final_mapping["review_status"] = np.where(
    candidate_mask,
    "confirmed_cumulus_ingredient_or_name",
    "name_screened_nonopioid"
)

# Distinguish ingredient evidence from product mapping.
df_final_mapping["mapping_scope"] = np.where(
    df_final_mapping["screening_method"] == "direct_cumulus",
    "direct_terminology_name",
    np.where(
        candidate_mask,
        "ingredient_level",
        "name_screening_only"
    )
)

# --------------------------------------------------
# 4. Integrity checks
# --------------------------------------------------

assert len(df_final_mapping) == 1034
assert df_final_mapping["record_count"].sum() == 54814

assert not df_final_mapping.duplicated(
    ["pyxis_name", "pyxis_gsn"]
).any()

assert df_final_mapping["is_cumulus_opioid"].notna().all()
assert df_final_mapping["is_cumulus_opioid"].isin([0, 1]).all()

assert df_final_mapping["is_cumulus_opioid"].sum() == 115

assert (
    df_final_mapping.loc[
        df_final_mapping["is_cumulus_opioid"] == 1,
        "record_count"
    ].sum() == 15851
)

# --------------------------------------------------
# 5. Export frozen mapping dictionary
# --------------------------------------------------

FINAL_MAPPING_PATH = (
    PROJECT_ROOT
    / "codebooks"
    / "opioid_medication_mapping_v1.csv"
)

FINAL_MAPPING_PATH.parent.mkdir(
    parents=True, exist_ok=True
)

df_final_mapping.to_csv(
    FINAL_MAPPING_PATH,
    index=False
)

# Confirm saved file can be reloaded correctly.
df_reloaded = pd.read_csv(
    FINAL_MAPPING_PATH,
    dtype={
        "pyxis_name": "string",
        "pyxis_gsn": "string",
        "is_cumulus_opioid": "Int64",
    }
)

assert len(df_reloaded) == 1034
assert df_reloaded["is_cumulus_opioid"].sum() == 115

print("--- Final Mapping Summary ---")

display(
    df_final_mapping.groupby(
        ["is_cumulus_opioid", "review_status"]
    ).agg(
        combinations=("pyxis_name", "size"),
        source_records=("record_count", "sum")
    ).reset_index()
)

print(f"\nFinal mapping saved: {FINAL_MAPPING_PATH}")
print("Final mapping integrity checks: PASSED")


--- Final Mapping Summary ---


,is_cumulus_opioid,review_status,combinations,source_records
0,0,name_screened_nonopioid,919,38963
1,1,confirmed_cumulus_ingredient_or_name,115,15851



Final mapping saved: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/codebooks/opioid_medication_mapping_v1.csv
Final mapping integrity checks: PASSED


#### Actual Results

The final Pyxis medication mapping dictionary was successfully generated and saved as `codebooks/opioid_medication_mapping_v1.csv`.

| Final Classification | Medication Combinations | Pyxis Source Records |
|---|---:|---:|
| Cumulus-related medication | 115 | 15,851 |
| Name-screened nonopioid | 919 | 38,963 |
| **Total** | **1,034** | **54,814** |

All 1,034 Pyxis name–GSN combinations were retained without duplicate keys. Final binary classifications contained no missing values, and the exported CSV passed reloading and integrity checks.

**Validation Status: PASSED**

#### Findings

The final mapping integrates direct Cumulus terminology matches, ingredient-level identification, targeted codeine review, and GSN-supported loperamide identification.

The mapping dictionary preserves the original Pyxis medication identifiers and matching evidence, allowing subsequent outcome construction to use a fixed, auditable classification.

The operational classification does not imply that every Pyxis product was independently mapped to an exact RxNorm clinical drug concept. Nonopioid classifications were based on the focused local-name review rather than comprehensive RxNorm verification.

#### Conclusion

The medication classification workflow is complete for the current study. The frozen mapping dictionary is ready for integration with Pyxis dispensing records during temporal attribution and encounter-level outcome construction in Section 5.4.

## 5.4 Temporal Attribution and Outcome Construction

### 5.4.1 Prepare the Frozen Medication Mapping for BigQuery

**Objective**

Prepare a minimal medication classification lookup for linking the frozen P5 medication mapping dictionary to MIMIC-IV-ED Pyxis records in BigQuery.

**Method**

1. Load the finalized medication mapping dictionary.
2. Retain the original Pyxis medication name, GSN, and binary Cumulus classification.
3. Verify unique mapping keys and complete classification values.
4. Preserve GSN identifiers as strings, including leading zeros.
5. Export a minimal CSV suitable for uploading to BigQuery.

The complete mapping dictionary remains the authoritative project reference. The exported lookup is a derived SQL integration artifact.

**Expected Output**

A validated medication lookup containing 1,034 unique Pyxis name–GSN combinations and their frozen binary classifications.

The lookup will be used to identify qualifying medication records without modifying the previously frozen P4 cohort.

In [12]:

# 5.4.1 Prepare Final Medication Lookup for BigQuery

mapping = pd.read_csv(
    FINAL_MAPPING_PATH,
    dtype={
        "pyxis_name": "string",
        "pyxis_gsn": "string",
        "is_cumulus_opioid": "Int64",
    }
)

df_opioid_lookup = mapping[
    [
        "pyxis_name",
        "pyxis_gsn",
        "is_cumulus_opioid",
    ]
].copy()

# Preserve the original medication identifiers.
assert len(df_opioid_lookup) == 1034

assert not df_opioid_lookup.duplicated(
    ["pyxis_name", "pyxis_gsn"]
).any()

assert df_opioid_lookup["pyxis_name"].notna().all()
assert df_opioid_lookup["is_cumulus_opioid"].notna().all()
assert df_opioid_lookup["is_cumulus_opioid"].isin([0, 1]).all()

assert df_opioid_lookup["is_cumulus_opioid"].sum() == 115

# Verify GSN preservation and missing values.
print("Missing GSN combinations:",
      df_opioid_lookup["pyxis_gsn"].isna().sum())

print("Missing-GSN opioid combinations:",
      df_opioid_lookup.loc[
          df_opioid_lookup["pyxis_gsn"].isna(),
          "is_cumulus_opioid"
      ].sum())

# Export minimal lookup for BigQuery.
LOOKUP_PATH = (
    PROJECT_ROOT
    / "results"
    / "05_opioid_outcome"
    / "opioid_mapping_bq_lookup.csv"
)

LOOKUP_PATH.parent.mkdir(
    parents=True, exist_ok=True
)

df_opioid_lookup.to_csv(
    LOOKUP_PATH,
    index=False
)

display(
    df_opioid_lookup.groupby("is_cumulus_opioid")
    .agg(combinations=("pyxis_name", "size"))
    .reset_index()
)

print(f"\nBigQuery lookup saved: {LOOKUP_PATH}")
print("5.4.1 Lookup preparation: PASSED")


Missing GSN combinations: 28
Missing-GSN opioid combinations: 1


,is_cumulus_opioid,combinations
0,0,919
1,1,115



BigQuery lookup saved: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/05_opioid_outcome/opioid_mapping_bq_lookup.csv
5.4.1 Lookup preparation: PASSED


### 5.4.2 BigQuery Medication Lookup and Opioid Record Extraction

#### 5.4.2.1 Validate the Uploaded Medication Lookup

**Objective**

Verify that the finalized medication classification lookup was imported correctly into BigQuery before joining it with MIMIC-IV-ED Pyxis records.

**Method**

**SQL Script:** `sql/05_opioid_outcome/03_validate_p5_mapping_lookup.sql`

1. Confirm the total number of medication name–GSN combinations.
2. Check for duplicate mapping keys and missing medication names.
3. Verify opioid and nonopioid classification counts.
4. Confirm that missing GSN values were preserved correctly.
5. Check whether GSN identifiers retained their leading zeros.

**Expected Output**

A validated BigQuery medication lookup containing 1,034 unique medication combinations, including 115 Cumulus-related and 919 name-screened nonopioid combinations.

The validated lookup will serve as the frozen medication classification reference for subsequent Pyxis extraction and temporal attribution.

#### Actual Results

The frozen medication mapping lookup was successfully imported into BigQuery and passed all structural validation checks.

| Validation Metric | Observed |
|---|---:|
| Total medication combinations | 1,034 |
| Unique name–GSN combinations | 1,034 |
| Cumulus-related combinations | 115 |
| Name-screened nonopioid combinations | 919 |
| Missing classifications | 0 |
| Missing medication names | 0 |
| Missing GSN combinations | 28 |
| Missing-GSN opioid combinations | 1 |
| GSN identifiers with leading zeros | 1,006 |

**Validation Status: PASSED**

#### Findings

The BigQuery lookup exactly reproduced the medication classification counts from the frozen Python mapping dictionary.

All medication name–GSN combinations remained unique, opioid classifications were complete, and leading zeros in GSN identifiers were preserved.

The 28 missing-GSN combinations, including one opioid-related candidate, require null-safe matching during Pyxis integration.

#### Conclusion

The BigQuery medication lookup is validated and ready for integration with Pyxis source records. Successful lookup validation does not yet establish complete source-record matching or correct temporal attribution; these will be assessed in the next step.

#### 5.4.2.2 Pyxis Mapping Integration and Temporal Distribution

**Objective**

Integrate the frozen Cumulus-based medication mapping with Pyxis records from the established P4 study population and examine the timing of opioid-related dispensing records relative to ED encounter boundaries.

**Method**

**SQL Script:** `sql/05_opioid_outcome/04_pyxis_opioid_temporal_audit.sql`

1. Restrict Pyxis records to the 14,293 frozen P4 ED encounters.
2. Join the medication mapping using the original Pyxis medication name and GSN, with null-safe handling of missing identifiers.
3. Verify source-record coverage and identify unmatched or multiply matched records.
4. Identify Cumulus-related medication records using the frozen binary classification.
5. Compare Pyxis `charttime` against ED `intime` and `outtime`.
6. Summarize record counts and distinct ED encounters by temporal category.

The candidate primary window is inclusive of both ED arrival and departure timestamps.

**Expected Output**

A source-level reconciliation and temporal distribution of Cumulus-related Pyxis records, including records documented during, before, and after the ED encounter.

These findings will inform the final temporal attribution rule and subsequent encounter-level outcome construction.

**Interpretation Boundary**

Pyxis records represent documented medication dispensing information. Their charting timestamps do not independently establish medication administration or ingestion.

#### Actual Results

The frozen Cumulus medication mapping was successfully joined to Pyxis records for the established P4 cohort.

All 54,814 Pyxis source rows were retained and classified, with no unmapped medication records.

| Validation Metric | Observed |
|---|---:|
| Total mapped Pyxis rows | 54,814 |
| Cumulus-related rows | 15,851 |
| Nonopioid rows | 38,963 |
| Unmapped rows | 0 |
| Missing-time opioid rows | 0 |

The temporal distribution of Cumulus-related records was:

| Timing Category | Source Rows | Distinct ED Encounters |
|---|---:|---:|
| Within ED | 14,830 | 6,552 |
| Before ED | 6 | 6 |
| After ED | 1,015 | 400 |

Of the 15,851 Cumulus-related source rows, 93.56% were charted within the ED stay, 6.40% after departure, and 0.04% before arrival.

#### Findings

**1. Complete medication mapping integration**

The BigQuery mapping reproduced the frozen medication dictionary totals, indicating that the medication-name and null-safe GSN join retained all source records without introducing duplicate matches.

**2. Temporal attribution**

A total of 6,646 unique ED encounters had at least one Cumulus-related Pyxis record at any charting time, compared with 6,552 encounters with at least one record within the ED stay.

Therefore, 94 encounters had qualifying medication records only outside the strict ED time window and would not be labeled positive under the primary outcome definition.

**3. Primary time-window decision**

The inclusive ED encounter window, `intime <= charttime <= outtime`, was selected for the primary outcome. This definition aligns the label with the documented ED encounter boundaries without introducing an arbitrary timing grace period.

#### Conclusion

Medication mapping integration and temporal attribution validation passed.

The primary outcome will identify ED encounters with at least one Cumulus-defined opioid-related Pyxis record charted within the inclusive ED stay window.

Records charted outside this window will not contribute to the primary binary label. Because Pyxis charting time may differ from the actual physical dispensing time, some timing-related misclassification remains possible.

### 5.4.3 Construct the Encounter-Level Opioid Dispensing Outcome

**Objective**

Generate one binary documented opioid dispensing outcome for each encounter in the frozen P4 study cohort using the validated medication mapping and finalized ED time-window definition.

**Method**
**SQL Script:** `sql/05_opioid_outcome/05_construct_opioid_outcome.sql`

1. Retrieve all 14,293 eligible ED encounters from the frozen P4 cohort lookup.
2. Identify Cumulus-related Pyxis records using the validated medication mapping.
3. Retain qualifying records whose charting timestamps fall within the inclusive ED encounter interval (`intime` to `outtime`).
4. Aggregate qualifying records at the `stay_id` level.
5. Assign `opioid_dispensed = 1` when at least one qualifying record exists; otherwise assign `0`.
6. Left join the outcome to the complete P4 cohort to preserve encounters with no qualifying records.

**Expected Output**

An encounter-level outcome table containing 14,293 unique `stay_id` values and a complete binary `opioid_dispensed` variable.

Based on the preceding temporal audit, 6,552 encounters are expected to receive a positive label.

**Interpretation**

The outcome represents documented Cumulus-defined opioid-related medication dispensing during the ED encounter. It does not establish medication administration, consumption, or clinical indication.

### 5.4.4 Integrate the Opioid Outcome with the Frozen P4 Cohort

**Objective**

Integrate the encounter-level opioid dispensing outcome with the frozen P4 analytical cohort while preserving the original study population and clinical features.

**Method**

1. Load the encounter-level outcome exported from BigQuery.
2. Validate unique encounter identifiers, binary outcome values, and completeness.
3. Merge the outcome into the frozen P4 dataset using `stay_id`, enforcing a one-to-one relationship.
4. Confirm that all P4 encounters and existing columns are preserved.
5. Verify C1 and C2 cohort membership and summarize the resulting opioid dispensing prevalence.
6. Export the combined dataset for subsequent outcome validation and predictive modeling.

**Expected Output**

A final encounter-level analytical dataset containing 14,293 ED encounters, the original 23 P4 columns, and one new binary outcome variable (`opioid_dispensed`).

Based on the preceding BigQuery temporal audit, 6,552 positive outcomes are expected, subject to confirmation from the actual exported outcome file.

**Interpretation**

The binary outcome represents documented Cumulus-defined opioid-related medication dispensing within the ED encounter window. It does not establish medication administration, opioid consumption, treatment indication, or clinical appropriateness.

In [13]:

# 5.4.4 Integrate Opioid Outcome with Frozen P4 Cohort

OUTCOME_PATH = (
    PROJECT_ROOT
    / "results"
    / "05_opioid_outcome"
    / "opioid_outcome_by_stay.csv"
)

FINAL_DATA_PATH = (
    PROJECT_ROOT
    / "results"
    / "05_opioid_outcome"
    / "adult_lbp_with_opioid_outcome.csv"
)

# --------------------------------------------------
# 1. Load encounter-level outcome
# --------------------------------------------------

assert OUTCOME_PATH.exists(), f"Missing file: {OUTCOME_PATH}"

df_outcome = pd.read_csv(
    OUTCOME_PATH,
    dtype={
        "stay_id": "string",
        "opioid_dispensed": "Int64",
    }
)

# Use a working copy to preserve the original P4 dataset
df_p4_work = df_p4.copy()

# Align join-key types
df_p4_work["stay_id"] = (
    df_p4_work["stay_id"].astype("string").str.strip()
)

df_outcome["stay_id"] = (
    df_outcome["stay_id"].str.strip()
)

# --------------------------------------------------
# 2. Validate encounter-level outcome
# --------------------------------------------------

assert len(df_outcome) == 14293
assert df_outcome["stay_id"].notna().all()
assert df_outcome["stay_id"].is_unique

assert df_outcome["opioid_dispensed"].notna().all()
assert df_outcome["opioid_dispensed"].isin([0, 1]).all()

print("--- Outcome Counts ---")

display(
    df_outcome["opioid_dispensed"]
    .value_counts()
    .sort_index()
    .rename_axis("opioid_dispensed")
    .reset_index(name="encounters")
)

# --------------------------------------------------
# 3. Validate frozen P4 dataset
# --------------------------------------------------

assert len(df_p4_work) == 14293
assert df_p4_work.shape[1] == 23

assert df_p4_work["stay_id"].notna().all()
assert df_p4_work["stay_id"].is_unique

assert "opioid_dispensed" not in df_p4_work.columns

# --------------------------------------------------
# 4. Merge outcome into frozen P4 cohort
# --------------------------------------------------

df_p5 = df_p4_work.merge(
    df_outcome[["stay_id", "opioid_dispensed"]],
    on="stay_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

# --------------------------------------------------
# 5. Integrity checks
# --------------------------------------------------

assert len(df_p5) == len(df_p4_work)
assert df_p5.shape[1] == 25  # Includes temporary _merge

assert (df_p5["_merge"] == "both").all()
assert df_p5["opioid_dispensed"].notna().all()
assert df_p5["stay_id"].is_unique

df_p5 = df_p5.drop(columns="_merge")

assert df_p5.shape == (14293, 24)

# Confirm all original P4 columns and values are preserved
pd.testing.assert_frame_equal(
    df_p5[df_p4_work.columns].reset_index(drop=True),
    df_p4_work.reset_index(drop=True),
    check_dtype=True
)

# --------------------------------------------------
# 6. Verify outcome counts and prevalence
# --------------------------------------------------

positive = int(df_p5["opioid_dispensed"].sum())
negative = int((df_p5["opioid_dispensed"] == 0).sum())

prevalence = positive / len(df_p5)

assert positive == 6552
assert negative == 7741
assert positive + negative == 14293

print("\n--- Final P5 Summary ---")
print(f"Total encounters: {len(df_p5):,}")
print(f"Unique patients: {df_p5['subject_id'].nunique():,}")
print(f"Total columns: {df_p5.shape[1]}")
print(f"Opioid-positive encounters: {positive:,}")
print(f"Opioid-negative encounters: {negative:,}")
print(f"Overall opioid dispensing prevalence: {prevalence:.2%}")

# --------------------------------------------------
# 7. Export final P5 analytical dataset
# --------------------------------------------------

FINAL_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_p5.to_csv(
    FINAL_DATA_PATH,
    index=False
)

print(f"\nFinal P5 dataset saved: {FINAL_DATA_PATH}")
print("P4-to-P5 integration checks: PASSED")


--- Outcome Counts ---


,opioid_dispensed,encounters
0,0,7741
1,1,6552



--- Final P5 Summary ---
Total encounters: 14,293
Unique patients: 11,404
Total columns: 24
Opioid-positive encounters: 6,552
Opioid-negative encounters: 7,741
Overall opioid dispensing prevalence: 45.84%

Final P5 dataset saved: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/05_opioid_outcome/adult_lbp_with_opioid_outcome.csv
P4-to-P5 integration checks: PASSED


#### Actual Results

The encounter-level opioid dispensing outcome was successfully integrated with the frozen P4 analytical cohort.

| Validation Metric | Observed |
|---|---:|
| Total ED encounters | 14,293 |
| Unique patients | 11,404 |
| Original P4 variables | 23 |
| Final P5 variables | 24 |
| Opioid-positive encounters | 6,552 |
| Opioid-negative encounters | 7,741 |
| Overall opioid dispensing prevalence | 45.84% |

The outcome counts matched those expected from the preceding BigQuery temporal audit.

#### Findings

The final dataset retained all 14,293 P4 encounters and 11,404 unique patients. One binary `opioid_dispensed` variable was appended without changing the original P4 analytical features.

The observed opioid-related dispensing prevalence was 45.84%, based on the frozen Cumulus medication mapping and the inclusive ED encounter time window.

This rate represents documented dispensing records rather than confirmed medication administration, patient consumption, or treatment appropriateness.

#### Conclusion

The P4-to-P5 outcome integration was successful. The resulting dataset is ready for cohort-specific prevalence analysis and final outcome validation before use in predictive modeling.

## 5.5 Outcome Validation

### 5.5.1 Cohort-Specific Opioid Dispensing Prevalence

**Objective**

Describe and compare documented opioid-related dispensing prevalence across the prespecified strict (C1) and expanded (C2) low back pain phenotypes.

**Method**

1. Use the finalized P5 analytical dataset.
2. Calculate encounter counts, opioid-positive counts, and prevalence for C1, C2-only, and the full C2 cohort.
3. Confirm that C1 and C2-only form a non-overlapping partition of C2.
4. Report descriptive differences without treating the overlapping C1 and C2 cohorts as independent groups.

**Expected Output**

A cohort-level summary of documented opioid dispensing prevalence, providing descriptive context for the phenotype sensitivity analysis in P6/P7.

**Interpretation**

Differences in observed prevalence may reflect variation in cohort composition, clinical presentation, or phenotype definition. They do not establish causal effects of ICD classification or opioid treatment decisions.

In [14]:

# 5.5.1 Inspect frozen cohort indicators

print("P5 columns:")
print(df_p5.columns.tolist())

print("\nPotential cohort columns:")
print([
    col for col in df_p5.columns
    if any(term in col.lower()
           for term in ["c1", "c2", "cohort", "phenotype"])
])


P5 columns:
['stay_id', 'subject_id', 'estimated_age', 'c1_flag', 'c2_flag', 'c2_only_flag', 'gender', 'arrival_transport', 'acuity', 'pain', 'heartrate', 'sbp', 'dbp', 'resprate', 'temperature', 'o2sat', 'chiefcomplaint', 'cc_text_normalized', 'cc_text_missing', 'cc_explicit_lbp', 'cc_sciatica', 'cc_broad_back_pain', 'cc_radiating_leg_pain', 'opioid_dispensed']

Potential cohort columns:
['c1_flag', 'c2_flag', 'c2_only_flag']


In [15]:

# 5.5.1 Cohort-Specific Opioid Dispensing Prevalence

# --------------------------------------------------
# 1. Validate cohort flags and outcome
# --------------------------------------------------

for col in ["c1_flag", "c2_flag", "c2_only_flag", "opioid_dispensed"]:
    assert df_p5[col].notna().all(), f"Missing values in {col}"
    assert df_p5[col].isin([0, 1]).all(), f"Invalid values in {col}"

# Verify mutually exclusive C1 / C2-only partition
assert (df_p5["c1_flag"] + df_p5["c2_only_flag"] == df_p5["c2_flag"]).all()

assert len(df_p5) == 14293
assert df_p5["c1_flag"].sum() == 9571
assert df_p5["c2_only_flag"].sum() == 4722
assert df_p5["c2_flag"].sum() == 14293

# --------------------------------------------------
# 2. Calculate cohort-specific prevalence
# --------------------------------------------------

cohort_definitions = {
    "C1 — Strict LBP": df_p5["c1_flag"] == 1,
    "C2-only — Additional": df_p5["c2_only_flag"] == 1,
    "C2 — Expanded LBP": df_p5["c2_flag"] == 1,
}

summary_rows = []

for cohort_name, mask in cohort_definitions.items():
    subset = df_p5.loc[mask]
    n = len(subset)
    positives = int(subset["opioid_dispensed"].sum())

    summary_rows.append({
        "Cohort": cohort_name,
        "Encounters": n,
        "Opioid-positive": positives,
        "Opioid-negative": n - positives,
        "Prevalence (%)": round(positives / n * 100, 2),
    })

df_cohort_prevalence = pd.DataFrame(summary_rows)

display(df_cohort_prevalence)

# --------------------------------------------------
# 3. Reconciliation checks
# --------------------------------------------------

c1_positive = df_cohort_prevalence.loc[
    df_cohort_prevalence["Cohort"] == "C1 — Strict LBP",
    "Opioid-positive"
].iloc[0]

c2_only_positive = df_cohort_prevalence.loc[
    df_cohort_prevalence["Cohort"] == "C2-only — Additional",
    "Opioid-positive"
].iloc[0]

c2_positive = df_cohort_prevalence.loc[
    df_cohort_prevalence["Cohort"] == "C2 — Expanded LBP",
    "Opioid-positive"
].iloc[0]

assert c1_positive + c2_only_positive == c2_positive
assert c2_positive == 6552

print("\nC1 + C2-only = C2: PASSED")
print("Overall positive count reconciliation: PASSED")

# Descriptive prevalence difference: C2-only versus C1
c1_rate = df_cohort_prevalence.loc[
    df_cohort_prevalence["Cohort"] == "C1 — Strict LBP",
    "Prevalence (%)"
].iloc[0]

c2_only_rate = df_cohort_prevalence.loc[
    df_cohort_prevalence["Cohort"] == "C2-only — Additional",
    "Prevalence (%)"
].iloc[0]

print(
    f"\nC2-only minus C1 prevalence difference: "
    f"{c2_only_rate - c1_rate:+.2f} percentage points"
)


,Cohort,Encounters,Opioid-positive,Opioid-negative,Prevalence (%)
0,C1 — Strict LBP,9571,4422,5149,46.20
1,C2-only — Additional,4722,2130,2592,45.11
2,C2 — Expanded LBP,14293,6552,7741,45.84



C1 + C2-only = C2: PASSED
Overall positive count reconciliation: PASSED

C2-only minus C1 prevalence difference: -1.09 percentage points


#### Actual Results

Documented opioid-related dispensing prevalence was calculated across the prespecified C1, C2-only, and full C2 cohorts.

| Cohort | ED Encounters | Opioid-Positive | Opioid-Negative | Prevalence |
|---|---:|---:|---:|---:|
| C1 — Strict LBP | 9,571 | 4,422 | 5,149 | 46.20% |
| C2-only — Additional | 4,722 | 2,130 | 2,592 | 45.11% |
| C2 — Expanded LBP | 14,293 | 6,552 | 7,741 | 45.84% |

All cohort membership and outcome reconciliation checks passed. C1 and C2-only formed a non-overlapping partition of C2, and their positive encounter counts summed to the overall total of 6,552.

#### Findings

**1. Similar observed prevalence across phenotype groups**

Documented opioid-related dispensing prevalence was 46.20% in C1 and 45.11% in the additional C2-only encounters, a descriptive difference of −1.09 percentage points.

The full expanded cohort (C2) had an overall prevalence of 45.84%.

**2. Limited prevalence sensitivity to cohort expansion**

Expanding the ICD-defined LBP cohort introduced 4,722 additional encounters without substantially changing the observed outcome prevalence.

This suggests that the primary outcome frequency was relatively stable across the two operational phenotype definitions in this dataset.

**3. Implications for downstream modeling**

Similar outcome prevalence does not establish equivalent predictive performance. Differences in clinical presentation, triage information, and predictor–outcome relationships may still influence model discrimination and calibration.

The planned C1-versus-C2 modeling experiment will examine these downstream effects.

#### Conclusion

Cohort-specific outcome prevalence analysis passed all integrity checks and showed a small descriptive difference between strict and additional expanded-phenotype encounters.

The results provide a baseline for the subsequent phenotype sensitivity analysis. They do not establish causal differences, statistical equivalence, or clinical appropriateness of opioid treatment.

### 5.5.2 Medication Composition Audit

**Objective**

Characterize the medications contributing to the documented Cumulus-defined opioid dispensing outcome and assess the clinical composition of qualifying Pyxis records.

**Method**
**SQL Script:** `sql/05_opioid_outcome/06_opioid_medication_composition.sql`
1. Identify Cumulus-positive Pyxis records within the finalized ED encounter time window.
2. Aggregate records by original Pyxis medication name and GSN.
3. Calculate source record counts and distinct ED encounter counts for each medication combination.
4. Review the most frequently documented medications and identify broader-scope opioid-related medications, including opioid antagonists and medications used for opioid use disorder.
5. Assess implications for interpretation of the primary outcome.

**Expected Output**

A medication-level frequency table showing the medications contributing to the primary dispensing outcome, with separate source-record and encounter counts.

**Interpretation Boundary**

A Pyxis source row is not necessarily an independent medication administration or dispensing event. The same ED encounter may involve multiple medications; therefore, medication-specific encounter counts are not additive across drug categories.

#### Actual Results

The medication composition audit identified 14,830 qualifying Cumulus-related Pyxis source records across 114 medication name–GSN combinations within the finalized ED encounter window.

The source record total reconciled exactly with the preceding temporal audit.

| Medication Group | Source Records |
|---|---:|
| Oxycodone-containing | 5,739 |
| Morphine-containing | 4,165 |
| Hydromorphone-containing | 3,528 |
| Tramadol-containing | 499 |
| Fentanyl-containing | 495 |
| Hydrocodone-containing | 269 |

These descriptive groups were derived from the original Pyxis medication names and are not intended to represent independent administration events.

#### Findings

**1. Predominance of commonly used opioid analgesics**

Oxycodone-, morphine-, and hydromorphone-containing medications accounted for 13,432 of the 14,830 qualifying source records (90.57%).

This indicates that the documented outcome was primarily represented by medications commonly used for analgesia, although individual treatment indications cannot be established from Pyxis records.

**2. Broad clinical scope of the Cumulus reference**

Less frequently documented medications included methadone (55 records), naloxone-containing products (26), buprenorphine-containing products (11), and loperamide (5).

The naloxone-containing count includes buprenorphine/naloxone combination products. These categories should not be interpreted as mutually exclusive or as evidence of specific treatment indications.

**3. Encounter-level interpretation remains necessary**

Medication-specific source record counts and distinct encounter counts cannot be added to estimate the overall number of positive ED encounters.

A focused encounter-level check is needed to determine whether broad-scope medications independently contribute positive outcome labels.

#### Conclusion

The medication composition audit reconciled with the finalized temporal attribution results and showed that commonly used opioid analgesics accounted for most qualifying Pyxis source records.

The presence of opioid antagonists and medications with non-analgesic uses reinforces the importance of describing the outcome as documented Cumulus-defined opioid-related medication dispensing, rather than confirmed analgesic treatment for low back pain.

### 5.5.3 Broad-Scope Medication Impact Check

**Objective**

Assess the contribution of selected broad-scope opioid-related medications to the final binary dispensing outcome.

**Method**

Using `sql/05_opioid_outcome/07_broad_scope_outcome_impact.sql`, qualifying Pyxis records were grouped by ED encounter to identify:

- Encounters involving naloxone-, loperamide-, methadone-, or buprenorphine-related medications.
- Encounters containing only these selected medications.
- Encounters also containing other qualifying opioids.

**Expected Output**

A descriptive encounter-level summary showing whether the selected broad-scope medication group materially contributes to positive outcome labels.

The primary outcome and frozen medication mapping remain unchanged.

#### Actual Results

The focused broad-scope medication impact audit identified 50 opioid-positive ED encounters containing at least one qualifying medication with a name matching naloxone, loperamide, methadone, or buprenorphine.

| Outcome Category | Encounters |
|---|---:|
| Total opioid-positive encounters | 6,552 |
| Encounters with broad-scope medications | 50 |
| Broad-scope-only encounters | 25 |
| Encounters with other qualifying opioids | 6,527 |

Of the 50 encounters involving broad-scope medications, 25 also contained another qualifying opioid medication, while 25 were positive exclusively through the selected broad-scope medication group.

The 25 broad-scope-only encounters represented 0.38% of all opioid-positive encounters and approximately 0.17% of the full analytical cohort.

#### Findings

**1. Limited contribution to positive outcome labels**

Broad-scope medications contributed relatively few exclusive positive encounters. Excluding the selected medication group would reduce the number of positive encounters from 6,552 to 6,527.

**2. Clinical interpretation**

The broad-scope category was identified through medication-name patterns and includes medications with potentially different clinical indications.

Methadone and buprenorphine may be used for pain management or opioid use disorder treatment, while naloxone and loperamide have other common clinical uses. Treatment indications cannot be determined from Pyxis medication names alone.

**3. Implications for outcome validity**

The small number of broad-scope-only encounters suggests that this selected medication group has a limited effect on the overall binary outcome prevalence.

However, this descriptive check does not establish equivalence of downstream model performance or eliminate other forms of medication mapping and timing-related misclassification.

#### Conclusion

The broad-scope medication audit supports retaining the prespecified Cumulus-defined outcome without modifying the frozen medication mapping or encounter labels.

The primary outcome remains documented dispensing of Cumulus-defined opioid-related medications, rather than confirmed opioid analgesic treatment for low back pain.


## 5.6 Final Summary and P6 Handoff

### Outcome Construction Summary

P5 successfully constructed an encounter-level binary outcome representing documented Cumulus-defined opioid-related medication dispensing during emergency department encounters.

The workflow used a frozen external Cumulus Opioid Value Set, a reviewed local Pyxis medication mapping dictionary, and an inclusive ED encounter time window (`intime <= charttime <= outtime`).

The final analytical dataset preserved the previously established P4 study population and appended one binary outcome variable, `opioid_dispensed`.

### Final Dataset

| Metric | Final Result |
|---|---:|
| ED encounters | 14,293 |
| Unique patients | 11,404 |
| Total variables | 24 |
| Opioid-positive encounters | 6,552 |
| Opioid-negative encounters | 7,741 |
| Overall dispensing prevalence | 45.84% |
| Missing outcome values | 0 |

### Cohort-Specific Outcomes

| Cohort | Encounters | Positive Encounters | Prevalence |
|---|---:|---:|---:|
| C1 — Strict LBP | 9,571 | 4,422 | 46.20% |
| C2-only — Additional | 4,722 | 2,130 | 45.11% |
| C2 — Expanded LBP | 14,293 | 6,552 | 45.84% |

The observed prevalence differed by 1.09 percentage points between C1 and C2-only encounters. This descriptive finding does not establish equivalent downstream predictive performance.

### Medication Mapping and Outcome Validation

The frozen mapping dictionary classified 115 Pyxis name–GSN combinations as Cumulus-related medications, representing 15,851 source records before temporal filtering.

Of these records, 14,830 were charted within the ED encounter window. The qualifying medication inventory was dominated by oxycodone-, morphine-, and hydromorphone-containing medications.

A focused broad-scope medication audit identified 25 positive encounters attributable exclusively to the selected group of naloxone-, loperamide-, methadone-, or buprenorphine-related medications. The primary outcome was retained without modification.

### Methodological Limitations

- Local medication mapping relied on direct terminology matches, ingredient-level name identification, and targeted review rather than complete product-level RxNorm normalization. Some medication misclassification remains possible.
- Pyxis charting timestamps do not necessarily represent the exact physical dispensing time and do not confirm medication administration or consumption.
- The Cumulus reference includes a broad range of opioid-related medications. The outcome must not be interpreted exclusively as opioid analgesic treatment for low back pain.
- The retrospective ICD-defined cohort is not directly identifiable at triage without later diagnostic information.

### P6 Modeling Handoff

**Final input dataset:**

`results/05_opioid_outcome/adult_lbp_with_opioid_outcome.csv`

**Prediction target:** `opioid_dispensed`

**Unit of analysis:** ED encounter (`stay_id`)

**Patient grouping:** `subject_id`

**Primary cohort:** C1 (`c1_flag == 1`)

**Sensitivity cohort:** C2 (`c2_flag == 1`)

P6 will use only information available at or before the selected triage prediction point. Final ICD-derived phenotype flags are used for cohort selection but must not enter the predictive feature matrix.

Medication names, Pyxis records, RxNorm identifiers, dispensing timestamps, and post-triage information must not be included as predictors.

Training and held-out test sets will be separated by `subject_id` to prevent patient-level leakage. The same patient assignment will be maintained across C1 and C2 experiments.

### Conclusion

P5 completed the construction, integration, and validation of the documented opioid dispensing outcome. The final dataset is ready for the planned predictive modeling and clinical phenotype sensitivity experiments in P6.